# Agent 1: Header-Only Naive Bayes Email Classifier
### A Regularized, Nested-CV Evaluated Component for Multi-Agent Phishing Detection

---

### Project Overview and Architecture
This notebook implements **Agent 1**, the email header analysis specialist in a three-agent cooperative architecture designed for robust email classification:
* **Agent 1 (This Notebook):** A regularized Naive Bayes classifier operating strictly on email metadata and header fields (`sender`, `receiver`, `date`, `subject`). It outputs three calibrated probabilities in the fixed order: `[p_safe, p_spam, p_phishing]`.
* **Agent 2:** A fine-tuned Transformer model operating strictly on the email body text.
* **Agent 3:** A Logistic Regression meta-classifier (stacking architecture) that learns optimal decision boundaries over the joint probability outputs of Agents 1 and 2.

### Scope and Isolation Guarantee
To ensure clear specialization and prevent multimodal contamination:
* Agent 1 has access **only** to the four agreed header fields: `sender`, `receiver`, `date`, and `subject`.
* Agent 1 **never inspects the email body**, URLs within the body, or attachments.
* Metadata fields such as `email_id`, `split`, and corpus `source` are strictly excluded from predictive features and used solely for grouping diagnostics and export alignment.

---

### Methodological Principles of this Implementation
1. **Mathematical Regularization:** Standard Naive Bayes assumes class-conditional independence, causing extreme probability overconfidence when applied to dozens of correlated header features. This implementation incorporates Laplace smoothing with empirical class priors, per-feature log-odds centering and clipping (`feat_bound`), and targeted missing-category regularization (`max_missing_log_odds`).
2. **Nested Sender-Disjoint Cross-Validation:** To prevent optimistic selection bias, all hyperparameter optimization (96 factorial configurations), post-hoc temperature calibration, and class decision threshold tuning are performed inside inner folds. The outer 5 folds evaluate the complete selection pipeline on entirely unseen sender identities.
3. **Template Memorization Audits:** Recurring subject templates across senders can artificially inflate accuracy. This notebook quantifies template overlap and stress-tests generalization under template-disjoint, template-purged, and connected-component evaluation regimes.
4. **Corpus Shortcut and Domain-Shift Diagnostics:** Features are audited for source identifiability. Leave-One-Source-Out (LOSO) cross-validation measures cross-corpus sensitivity and guards against learning collection artifacts.
5. **Strict Cross-Fitting for Stacking:** Training predictions for Agent 3 (`agent1_oof_train.csv`) are generated via strict out-of-fold cross-fitting with optional template purging, guaranteeing zero target leakage or tuning look-ahead for the meta-classifier.

---

### Evaluation Hierarchy Guide
When reviewing the empirical results in this notebook, results should be interpreted in the following order of scientific confidence:
1. **Nested Sender-Disjoint CV (Section 7–8):** The headline performance estimate for generalization to novel senders.
2. **Template-Disjoint / Purged Diagnostics (Section 9):** Stress tests measuring performance when subject templates are novel.
3. **Leave-One-Source-Out (Section 10):** Diagnostic measuring robustness against inter-corpus distribution shift.
4. **Validation Split (Section 15):** Evaluated strictly once on the frozen model; validation labels are never used for tuning.
5. **Inner-CV Selection Tables:** Intermediate metrics used solely to guide candidate ranking; optimistic by construction.

In [5]:
# 2. Imports and configuration
import os, sys, glob, json, hashlib, pathlib, subprocess, warnings, random, time, platform
import numpy as np, pandas as pd, joblib, sklearn, scipy
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, log_loss, average_precision_score, accuracy_score, precision_recall_curve,
                             confusion_matrix, precision_recall_fscore_support)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------ reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# ------------------------------------------------------------------ experiment switches
N_SPLITS = 5                      # outer folds / final grouped CV
N_INNER  = 4                      # inner folds used inside every outer training set
QUICK_SEARCH = os.environ.get("AGENT1_QUICK", "0") == "1"   # True = much smaller candidate grids (for a first run / debugging)
NESTED_GRID_MODE = "full"         # the 96-candidate grid is searched INDEPENDENTLY inside every outer training fold (no lik_scale fixed in advance). "reduced" exists only to reproduce the previous methodology for the comparison in section 13b
N_JOBS = int(os.environ.get("AGENT1_JOBS", max(1, min(4, os.cpu_count() or 1))))

RUN_TEMPLATE_EXPERIMENTS = True   # section 11
RUN_REDUCED_COMPARISON = True     # section 13b: re-run the previous reduced-grid nested CV (same folds, same seeds) to compare old vs new methodology
OOF_PURGE_TEMPLATES = True        # Agent-3 training predictions: also hide the held-out rows' subject templates from the fitting/selection data (strictest). False = sender-grouped only (closer to serving conditions)
USE_CACHE = True                  # checkpoint every finished outer fold in OUT/cache so an interrupted run can resume (keys include data + code hashes)
RUN_LOSO   = True                 # section 12
RUN_ABLATION = True               # sections 11-12 (feature-group ablation under 3 protocols)

# decision rule used for the exported `pred_class` ("argmax" | "macro_f1" | "recall_at_precision")
OFFSET_MODE = "macro_f1"
MIN_PHISH_PRECISION = 0.70        # only used when OFFSET_MODE == "recall_at_precision"
PRECISION_FLOORS = (0.60, 0.65, 0.70, 0.75, 0.80)   # operating points compared in section 10

FINAL_CFG_OVERRIDE = None         # e.g. {"feature_set": "compact_no_clock"} to force a feature set in the final model (then T/offsets are re-fitted)

REPORT_ON_VAL  = True             # labels are read only AFTER the model is frozen
REPORT_ON_TEST = False            # test was seen in earlier iterations -> not a pristine hold-out

OUT = pathlib.Path("agent1_final_outputs"); OUT.mkdir(exist_ok=True); (OUT / "cache").mkdir(exist_ok=True)
HEADER_COLS = ["sender", "receiver", "date", "subject"]
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_colwidth", 60)
print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | sklearn {sklearn.__version__} | scipy {scipy.__version__}")
print(f"QUICK_SEARCH={QUICK_SEARCH} NESTED_GRID_MODE={NESTED_GRID_MODE} N_JOBS={N_JOBS} OFFSET_MODE={OFFSET_MODE}")
if QUICK_SEARCH: print("!! QUICK_SEARCH is ON: grids are smaller than the 96-candidate search. Results are NOT the full nested search and must not be reported as such.")

python 3.13.18 | numpy 2.1.3 | pandas 2.3.3 | sklearn 1.6.1 | scipy 1.16.3
QUICK_SEARCH=False NESTED_GRID_MODE=full N_JOBS=4 OFFSET_MODE=macro_f1


## 1. Model Architecture and Embedded Inference Module

This section encapsulates the complete Agent 1 model within a self-contained source string (`HEADER_SRC`). At runtime, this module is written to disk (`agent1_header.py`) and dynamically imported. This design ensures that:
1. Multiprocessing workers (`joblib.Parallel`) can import the exact model classes across subprocesses.
2. The serialized artifact (`agent1_model.joblib`) can be loaded and executed in a fresh, isolated Python environment without requiring notebook session state.

### Key Algorithmic Components:
* **MIME Decoding and Address Normalization:** Bounded decoding handling multi-part headers, obfuscated encodings, and address normalization (`getaddresses` with regular expression fallback).
* **Canonical Sender Identification:** Senders from custom domains are indexed by their registrable domain (`dom:domain.tld`), while free-mail and IP senders are indexed by full address (`addr:user@host.tld`).
* **Feature Extraction:** Extracts 36 engineered signals covering sender structure, domain spoofing/lookalike heuristics, brand token matches, recipient counts, timezone offsets, and subject lexical features.
* **HeaderPreprocessor:** Quantile discretization computed strictly on non-abstained values, TLD vocabulary building, and explicit missingness category encoding.
* **SmoothedCategoricalNB:** Custom regularized Categorical Naive Bayes incorporating:
  * Empirical class prior probabilities.
  * Feature log-odds median centering and bounding to $[-b, +b]$ (`feat_bound`).
  * Targeted missing-category log-odds clipping (`max_missing_log_odds`).
  * Explicit abstention (zero log-likelihood contribution) for derived features whose parent header is absent.
* **SubjectTokenNB:** Unigram subject lexical model providing complementary token evidence.
* **HeaderAgent:** Top-level estimator providing `fit_features()`, `proba_features()`, post-hoc temperature calibration, and decision offset adjustments.

In [6]:
# 3. Agent 1 model code (feature extractor + Naive Bayes). Everything lives in this notebook.
HEADER_SRC = r'''
"""
agent1_header.py - Header-only Naive Bayes agent (v7.0.0)

Inputs : sender, receiver, date, subject (raw header strings). Never the body.
Outputs: P(safe), P(spam), P(phishing) in this fixed order (0=safe, 1=spam, 2=phishing).

Changes vs v4 (see notebook cell 0 for the audit IDs):
  B2  weekday is computed from the parsed y/m/d (parsedate_tz always returns 0 for it)
  B3  unparseable dates abstain for hour/weekday; timezone abstains unless an explicit offset exists
  B4  "undisclosed-recipients" is its own state instead of being erased as "missing"
  B5  brand-typo detection needs tokens >= 5 chars, skips common words, never fuzzes short brands;
      brand spoofing in sub-domains (paypal.com.evil.ru) is detected; IPv4 / punycode hosts handled
  N1  likelihoods are conditioned on NON-abstained rows (v4 counted abstained rows in the
      denominator, which double-counted missingness and biased class likelihoods)
  M4  optional per-group feature weights to limit double counting of correlated features
  M5  temperature calibration + class-prior offsets are stored in the agent (fitted outside, on OOF only)
  M6  optional subject-token Naive Bayes block (centered, clipped, length-normalised)
  v7  Q1  quantile bin edges are fitted on NON-abstained rows only (placeholder zeros of missing headers used to pollute them)
      Q2  r_same_domain_as_sender abstains when the sender is missing (it used to report a fake "different domain")
      Q3  offset search split into offset_table() / pick_offsets() so the same table serves several decision rules
"""
import re, datetime
from email.header import decode_header, make_header
from email.utils import getaddresses, parsedate_tz

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize_scalar

__version__ = "7.0.0"

CLASS_NAMES = ("safe", "spam", "phishing")
N_CLASSES = 3
PROB_FLOOR = 1e-6
MAX_RAW = 4096
MAX_HDR = 512

_PREFIX_RE = re.compile(r"^\s*(?:re|fwd?|fw|antw|wg|tr|aw)\s*[:\-]\s*", re.I)
_ADDR_RE = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
_IPV4_RE = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")
_TZ_TAIL_RE = re.compile(r"\b([A-Z]{1,4})\b\s*$")
_TZ_NUM_RE = re.compile(r"\d:\d{2}(?::\d{2})?\s*([+-]\d{4})\b")

FREE_MAIL = frozenset({
    "gmail.com", "yahoo.com", "hotmail.com", "aol.com", "outlook.com",
    "mail.com", "msn.com", "live.com", "ymail.com", "comcast.net",
    "sbcglobal.net", "verizon.net", "att.net", "bellsouth.net",
    "rediffmail.com", "cox.net", "charter.net", "shaw.ca", "earthlink.net"})

BRAND_TOKENS = frozenset({
    "paypal", "ebay", "amazon", "apple", "microsoft", "google",
    "chase", "wellsfargo", "bankofamerica", "citi", "citigroup",
    "barclays", "hsbc", "netflix", "irs", "usps", "fedex", "ups",
    "dhl", "facebook", "instagram", "whatsapp", "linkedin", "twitter"})
# Fuzzy (one-edit) matching only for brands with >= 5 characters; short brands are exact-match only.
FUZZY_BRANDS = frozenset(b for b in BRAND_TOKENS if len(b) >= 5)
COMMON_WORDS = frozenset({"apply", "ample", "maple", "grape", "phase", "chose", "chasm", "apples"})

SUSPICIOUS_TLDS = frozenset({
    "tk", "ml", "ga", "cf", "gq", "xyz", "top", "work", "loan",
    "click", "fit", "surf", "buzz", "racing", "date", "faith"})

KNOWN_TWO_LEVEL_TLDS = frozenset({
    "co.uk", "org.uk", "me.uk", "ltd.uk", "plc.uk", "net.uk",
    "com.au", "net.au", "org.au", "edu.au", "gov.au",
    "co.nz", "net.nz", "org.nz", "ac.nz", "govt.nz",
    "co.za", "org.za", "net.za", "gov.za",
    "co.jp", "ne.jp", "or.jp", "go.jp", "ac.jp",
    "com.br", "net.br", "org.br", "gov.br",
    "com.mx", "org.mx", "net.mx", "edu.mx", "gob.mx",
    "com.cn", "net.cn", "org.cn", "gov.cn", "edu.cn",
    "co.in", "net.in", "org.in", "gen.in", "firm.in",
    "gc.ca", "fed.us"})

RFC_TZ_OFFSETS = {
    "UT": 0, "UTC": 0, "GMT": 0, "Z": 0,
    "EDT": -4, "EST": -5, "CDT": -5, "CST": -6,
    "MDT": -6, "MST": -7, "PDT": -7, "PST": -8,
    "A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6,
    "G": 7, "H": 8, "I": 9, "K": 10, "L": 11, "M": 12,
    "N": -1, "O": -2, "P": -3, "Q": -4, "R": -5, "S": -6,
    "T": -7, "U": -8, "V": -9, "W": -10, "X": -11, "Y": -12}


# ----------------------------------------------------------------------------- parsing helpers
def _decode(v, limit=MAX_HDR):
    if isinstance(v, bytes):
        v = v.decode("utf-8", "replace")
    if not isinstance(v, str):
        return ""
    raw = v[:MAX_RAW]
    try:
        dec = str(make_header(decode_header(raw)))
    except Exception:
        dec = raw
    return dec[:limit].strip()


def _is_ip(host):
    return bool(_IPV4_RE.match(str(host).strip("[] ")))


def registrable_domain(domain):
    if not domain or not isinstance(domain, str):
        return ""
    d = domain.strip().lower().strip(". >)(<[]")
    if _IPV4_RE.match(d):
        return d
    parts = d.split(".")
    if len(parts) <= 2:
        return d
    if ".".join(parts[-2:]) in KNOWN_TWO_LEVEL_TLDS:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:])


def _normalize_subject_text(dec):
    txt = dec.lower()
    for _ in range(5):
        new = _PREFIX_RE.sub("", txt).strip()
        if new == txt:
            break
        txt = new
    txt = re.sub(r"\d+", "#", txt)
    txt = re.sub(r"[^\w\s#]", " ", txt)
    return " ".join(txt.split())


def normalize_subject(s):
    return _normalize_subject_text(_decode(s))


def _parse_addrs(text):
    out = []
    if not text or not isinstance(text, str):
        return out
    try:
        try:
            items = getaddresses([text], strict=False)
        except TypeError:
            items = getaddresses([text])
        out = [(n.strip(), a.strip().lower()) for n, a in items if a and "@" in a]
    except Exception:
        out = []
    if not out:
        out = [("", a.lower()) for a in _ADDR_RE.findall(text)]
    return out


def _within_one_edit(a, b):
    if a == b:
        return False
    la, lb = len(a), len(b)
    if abs(la - lb) > 1:
        return False
    if la == lb:
        return sum(c1 != c2 for c1, c2 in zip(a, b)) == 1
    short, long_ = (a, b) if la < lb else (b, a)
    return any(long_[:i] + long_[i + 1:] == short for i in range(len(long_)))


def canonical_sender_identity(dom, addr):
    """CV grouping key: exact address for free-mail / IP / unknown domains, registrable domain otherwise."""
    if not addr or "@" not in str(addr):
        return None
    d = str(dom or "").strip().lower()
    if not d or d in FREE_MAIL or _is_ip(d):
        return f"addr:{str(addr).strip().lower()}"
    return f"dom:{d}"


def _ensure_df(x):
    if isinstance(x, pd.DataFrame):
        return x
    if isinstance(x, dict):
        return pd.DataFrame([x])
    if isinstance(x, (list, tuple)):
        return pd.DataFrame(list(x))
    raise TypeError(type(x))


# ----------------------------------------------------------------------------- per-field extraction
_SENDER_EMPTY = dict(
    s_missing=1, s_addr="", s_dom="", s_tld="", s_dom_levels=0, s_has_num=0, s_has_hyphen=0,
    s_is_freemail=0, s_suspicious_tld=0, s_brand_in_domain=0, s_brand_spoof_host=0, s_typo_brand=0,
    s_ip_domain=0, s_punycode=0, s_disp_len=0, s_disp_has_excl=0, s_disp_has_quest=0,
    s_disp_has_at=0, s_disp_has_dom=0)


def _sender_info(v):
    row = dict(_SENDER_EMPTY)
    addrs = _parse_addrs(_decode(v))
    if not addrs:
        return row
    disp, a = addrs[0]
    host = a.split("@", 1)[1].strip(".[]> <)(") if "@" in a else ""
    if not host:
        return row
    row["s_missing"] = 0
    row["s_addr"] = a
    labels = host.split(".")
    row["s_dom_levels"] = len(labels)
    row["s_disp_len"] = len(disp)
    row["s_disp_has_excl"] = int("!" in disp)
    row["s_disp_has_quest"] = int("?" in disp)
    row["s_disp_has_at"] = int("@" in disp)
    row["s_disp_has_dom"] = int(bool(re.search(r"\b[a-z0-9-]+\.[a-z]{2,}\b", disp, re.I)))
    row["s_punycode"] = int(any(l.startswith("xn--") for l in labels))
    if _is_ip(host):
        row["s_dom"] = host.strip("[]")
        row["s_ip_domain"] = 1
        return row
    reg = registrable_domain(host)
    tld = reg.split(".")[-1] if "." in reg else ""
    row["s_dom"], row["s_tld"] = reg, tld
    row["s_has_num"] = int(bool(re.search(r"\d", reg)))
    row["s_has_hyphen"] = int("-" in reg)
    row["s_is_freemail"] = int(reg in FREE_MAIL)
    row["s_suspicious_tld"] = int(tld in SUSPICIOUS_TLDS)
    reg_tokens = {t for t in re.split(r"[^a-z0-9]+", reg) if t}
    sub_part = host[: len(host) - len(reg)] if host.endswith(reg) else ""
    sub_tokens = {t for t in re.split(r"[^a-z0-9]+", sub_part) if t}
    in_reg = reg_tokens & BRAND_TOKENS
    row["s_brand_in_domain"] = int(bool(in_reg))
    # brand named in a sub-domain of an unrelated registrable domain, e.g. paypal.com.evil.ru
    row["s_brand_spoof_host"] = int(bool((sub_tokens & BRAND_TOKENS) - in_reg))
    row["s_typo_brand"] = int(any(
        len(t) >= 5 and t not in COMMON_WORDS and t not in BRAND_TOKENS and _within_one_edit(t, b)
        for t in reg_tokens for b in FUZZY_BRANDS))
    return row


def _receiver_info(v, sdom):
    dec = _decode(v)
    addrs = _parse_addrs(dec)
    undis = int("undisclosed" in dec.lower())
    row = dict(r_missing=0, r_state=0, r_count=0, r_disp_len=0, r_disp_has_at=0, r_has_undisclosed=undis,
               r_same_domain_as_sender=0, r_is_freemail=0)
    if not addrs:
        row["r_missing"] = 0 if undis else 1     # B4: "undisclosed-recipients:;" is a real state, not "missing"
        row["r_state"] = 1 if undis else 0       # 0 missing, 1 undisclosed, 2 single recipient, 3 several
        return row
    disp, a = addrs[0]
    host = a.split("@", 1)[1] if "@" in a else ""
    reg = registrable_domain(host)
    row.update(r_state=1 if undis else (2 if len(addrs) == 1 else 3),
               r_count=len(addrs), r_disp_len=len(disp), r_disp_has_at=int("@" in disp),
               r_same_domain_as_sender=int(reg == sdom and bool(sdom)), r_is_freemail=int(reg in FREE_MAIL))
    return row


def _date_info(v):
    """-> (status, hour, weekday, tz_hours). status: 0 missing, 1 parsed w/o tz, 2 parsed with tz, 3 unparseable."""
    dec = _decode(v)
    if not dec:
        return 0, -1, -1, 0
    try:
        pt = parsedate_tz(dec)
    except Exception:
        pt = None
    if pt is None:
        return 3, -1, -1, 0
    y, mo, d, hr = pt[0], pt[1], pt[2], pt[3]
    if not (0 <= hr <= 23):
        return 3, -1, -1, 0
    try:
        wd = datetime.date(y, mo, d).weekday()      # B2: parsedate_tz()[6] is always 0, so compute it
    except Exception:
        return 3, -1, -1, 0
    # parsedate_tz returns offset 0 (not None) when no zone is present, so require an explicit zone token.
    body = re.sub(r"\([^)]*\)\s*$", "", dec).strip()
    if _TZ_NUM_RE.search(body):
        return 2, hr, wd, int(round((pt[9] or 0) / 3600.0))
    m = _TZ_TAIL_RE.search(body)
    if m and m.group(1) in RFC_TZ_OFFSETS:
        return 2, hr, wd, RFC_TZ_OFFSETS[m.group(1)]
    return 1, hr, wd, 0


def _subject_info(v):
    dec = _decode(v)
    if not dec:
        return dict(u_missing=1, u_len=0, u_num_words=0, u_caps_ratio=0.0, u_num_punct=0,
                    u_num_digits=0, u_is_fwd=0, u_is_reply=0, u_norm="")
    letters = [c for c in dec if c.isalpha()]
    return dict(
        u_missing=0, u_len=len(dec), u_num_words=len(dec.split()),
        u_caps_ratio=(sum(c.isupper() for c in letters) / len(letters)) if letters else 0.0,
        u_num_punct=sum(c in "!?;:#$%&'\"" for c in dec),
        u_num_digits=sum(c.isdigit() for c in dec),
        u_is_fwd=int(bool(re.search(r"^\s*(?:fwd?|fw)\s*[:\-]", dec, re.I))),
        u_is_reply=int(bool(re.search(r"^\s*(?:re|antw|wg|tr|aw)\s*[:\-]", dec, re.I))),
        u_norm=_normalize_subject_text(dec))


def extract_features(df):
    df = _ensure_df(df)
    N = len(df)
    col = lambda name: df[name].tolist() if name in df else [None] * N
    S = pd.DataFrame([_sender_info(v) for v in col("sender")], index=df.index)
    R = pd.DataFrame([_receiver_info(v, d) for v, d in zip(col("receiver"), S["s_dom"].tolist())], index=df.index)
    D = pd.DataFrame([_date_info(v) for v in col("date")], index=df.index,
                     columns=["d_date_status", "d_hour", "d_weekday", "d_tz_h"])
    U = pd.DataFrame([_subject_info(v) for v in col("subject")], index=df.index)
    return pd.concat([S, R, D, U], axis=1)


# ----------------------------------------------------------------------------- feature specification
SPEC = {
    "s_missing": ("bin",), "s_dom_levels": ("cap", 6), "s_has_num": ("bin",), "s_has_hyphen": ("bin",),
    "s_is_freemail": ("bin",), "s_suspicious_tld": ("bin",), "s_brand_in_domain": ("bin",),
    "s_brand_spoof_host": ("bin",), "s_typo_brand": ("bin",), "s_ip_domain": ("bin",), "s_punycode": ("bin",),
    "s_disp_len": ("quant",), "s_disp_has_excl": ("bin",), "s_disp_has_quest": ("bin",),
    "s_disp_has_at": ("bin",), "s_disp_has_dom": ("bin",), "s_tld": ("cat",),
    "r_missing": ("bin",), "r_state": ("cap", 3), "r_count": ("cap", 5), "r_disp_len": ("quant",), "r_disp_has_at": ("bin",),
    "r_has_undisclosed": ("bin",), "r_same_domain_as_sender": ("bin",), "r_is_freemail": ("bin",),
    "d_date_status": ("cap", 3), "d_hour": ("edges", list(range(0, 24, 2))),
    "d_weekday": ("cap", 6), "d_tz_h": ("edges", list(range(-12, 14, 2))),
    "u_missing": ("bin",), "u_len": ("quant",), "u_num_words": ("quant",), "u_caps_ratio": ("quant",),
    "u_num_punct": ("quant",), "u_num_digits": ("quant",), "u_is_fwd": ("bin",), "u_is_reply": ("bin",)}

FEATURE_GROUPS = {g: [f for f in SPEC if f.startswith(p)]
                  for g, p in (("sender", "s_"), ("receiver", "r_"), ("date", "d_"), ("subject", "u_"))}
# "legacy" receiver block = 7 near-duplicate features (all mostly encode "receiver hidden/absent"); r_state replaces them in the compact sets
ALL_FEATURES = [f for g in ("sender", "receiver", "date", "subject") for f in FEATURE_GROUPS[g] if f != "r_state"]
COMPACT_RECEIVER = ["r_state", "r_same_domain_as_sender", "r_is_freemail"]
ALL_FEATURES_COMPACT = [f for f in ALL_FEATURES if not f.startswith("r_")] + COMPACT_RECEIVER
MISSINGNESS_FEATURES = ["s_missing", "r_missing", "r_state", "u_missing", "d_date_status"]
CLOCK_FEATURES = ["d_hour", "d_weekday", "d_tz_h"]          # era / locale proxies; ablated in model selection


class HeaderPreprocessor:
    def __init__(self, features, n_bins=24, top_tld=20):
        self.features = list(features)
        self.n_bins, self.top_tld = n_bins, top_tld

    def fit(self, Xf):
        self.edges_, self.tlds_ = {}, []
        M = self._masks(Xf)
        for f in self.features:
            kind = SPEC[f][0]
            if kind == "quant":
                v = pd.to_numeric(Xf[f], errors="coerce").to_numpy(dtype=float)
                am = self.abstain_mask(f, M)                  # Q1: only rows that carry evidence define the bins
                if am is not None:
                    v = v[~am]
                v = v[~np.isnan(v)]
                self.edges_[f] = (np.unique(np.quantile(v, np.linspace(0, 1, self.n_bins + 1)[1:-1])).tolist()
                                  if len(v) else [])
            elif kind == "edges":
                self.edges_[f] = list(SPEC[f][1])
        if "s_tld" in self.features and self.top_tld > 0:
            vc = Xf["s_tld"].astype(str).replace("nan", "").value_counts()
            self.tlds_ = [t for t in vc.index if t][: self.top_tld]
        self.n_categories_ = [self._k(f) for f in self.features]
        return self

    def _k(self, f):
        kind = SPEC[f][0]
        if kind == "bin":
            return 3                                    # 0, 1, abstain
        if kind == "cap":
            return SPEC[f][1] + 2                       # 0..cap, abstain
        if kind in ("quant", "edges"):
            return len(self.edges_[f]) + 3              # bins, missing, abstain
        if kind == "cat":
            return len(self.tlds_) + 3                  # known, other, none, abstain
        raise ValueError(kind)

    def missing_category(self, f):
        if f in ("s_missing", "r_missing", "u_missing"):
            return 1
        if f in ("d_date_status", "r_state"):
            return 0
        if SPEC[f][0] in ("quant", "edges", "cat"):
            return self._k(f) - 2
        return None

    @staticmethod
    def _masks(Xf):
        n = len(Xf)
        z = np.zeros(n, bool)
        g = lambda c: Xf[c].to_numpy() if c in Xf.columns else None
        s_m = g("s_missing"); s_m = s_m.astype(bool) if s_m is not None else z
        r_m = g("r_missing"); r_m = r_m.astype(bool) if r_m is not None else z
        u_m = g("u_missing"); u_m = u_m.astype(bool) if u_m is not None else z
        rc = g("r_count"); r_noaddr = (rc == 0) if rc is not None else z
        st = g("d_date_status")
        d_bad = np.isin(st, (0, 3)) if st is not None else z
        d_notz = (st != 2) if st is not None else z
        return dict(s=s_m, r=r_m, u=u_m, r_noaddr=np.asarray(r_noaddr, bool),
                    d_bad=np.asarray(d_bad, bool), d_notz=np.asarray(d_notz, bool))

    @staticmethod
    def abstain_mask(f, M):
        """Rows for which feature f carries no evidence (parent header absent / unparseable)."""
        if f in MISSINGNESS_FEATURES:
            return None
        if f.startswith("s_"):
            return M["s"]
        if f.startswith("r_"):
            if f == "r_same_domain_as_sender":              # Q2: meaningless without a sender domain
                return M["r"] | M["r_noaddr"] | M["s"]
            return M["r"] if f in ("r_count", "r_has_undisclosed") else (M["r"] | M["r_noaddr"])
        if f.startswith("u_"):
            return M["u"]
        if f.startswith("d_"):
            return M["d_notz"] if f == "d_tz_h" else M["d_bad"]
        return None

    def transform(self, Xf):
        M = self._masks(Xf)
        cols = []
        for f in self.features:
            K = self._k(f)
            kind = SPEC[f][0]
            if kind == "bin":
                c = np.clip(pd.to_numeric(Xf[f], errors="coerce").fillna(0).to_numpy(dtype=int), 0, 1)
            elif kind == "cap":
                c = np.clip(pd.to_numeric(Xf[f], errors="coerce").fillna(0).to_numpy(dtype=int), 0, SPEC[f][1])
            elif kind in ("quant", "edges"):
                ed = self.edges_[f]
                v = pd.to_numeric(Xf[f], errors="coerce").to_numpy(dtype=float)
                nan = np.isnan(v)
                c = np.searchsorted(ed, np.nan_to_num(v, nan=0.0), side="right")
                c[nan] = len(ed) + 1
            else:  # cat
                m = {t: i for i, t in enumerate(self.tlds_)}
                s = Xf[f].astype(str).str.lower()
                c = np.array([len(self.tlds_) + 1 if (not x or x == "nan") else m.get(x, len(self.tlds_))
                              for x in s], dtype=int)
            am = self.abstain_mask(f, M)
            if am is not None:
                c = np.where(am, K - 1, c)
            cols.append(c)
        return np.column_stack(cols).astype(np.int32)


# ----------------------------------------------------------------------------- models
class SmoothedCategoricalNB:
    """Categorical NB. Likelihoods are conditioned on non-abstained rows (N1 fix).
    smoothing='marginal': Dirichlet prior proportional to the pooled marginal; 'laplace': uniform (== sklearn CategoricalNB).
    Median-centering + clipping (feat_bound, max_missing_log_odds) and feature weights make it a *regularised* NB."""

    def __init__(self, alpha=1.0, lik_scale=1.0, internal_4class=False, max_missing_log_odds=1.5,
                 feat_bound=2.5, smoothing="marginal"):
        self.alpha, self.lik_scale = float(alpha), float(lik_scale)
        self.internal_4class = bool(internal_4class)
        self.max_missing_log_odds = float(max_missing_log_odds)
        self.feat_bound = float(feat_bound) if feat_bound is not None else 0.0
        self.smoothing = smoothing

    def fit(self, X, y, n_categories, preprocessor=None, feature_weights=None):
        y = np.asarray(y, dtype=int)
        C = 4 if self.internal_4class else N_CLASSES
        self.n_internal_ = C
        cc = np.bincount(y, minlength=C).astype(float)
        if (cc == 0).any():
            raise ValueError(f"Training data must contain all {C} classes; counts={cc.tolist()}")
        self.log_prior_ = np.log(cc / cc.sum())
        self.prior_probs_ = cc / cc.sum()
        self.weights_ = np.ones(len(n_categories)) if feature_weights is None else np.asarray(feature_weights, float)
        self.tables_ = []
        for f, K in enumerate(n_categories):
            col = X[:, f]
            real = col != (K - 1)
            cnt = np.vstack([np.bincount(col[(y == c) & real], minlength=K)[: K - 1] for c in range(C)]).astype(float)
            n_real = cnt.sum(1)
            tot = cnt.sum(0)
            if self.smoothing == "laplace":
                p = (cnt + self.alpha) / (n_real[:, None] + self.alpha * (K - 1))
            else:
                marg = (tot + 1.0) / (tot.sum() + (K - 1))
                p = (cnt + self.alpha * marg) / (n_real[:, None] + self.alpha)
            ll = np.zeros((K, C))
            ll[: K - 1] = np.log(p).T
            ll[: K - 1][tot == 0] = 0.0
            if self.feat_bound > 0.0:
                for k in range(K - 1):
                    ll[k] = np.clip(ll[k] - np.median(ll[k]), -self.feat_bound, self.feat_bound)
            if self.max_missing_log_odds > 0.0 and preprocessor is not None:
                miss = preprocessor.missing_category(preprocessor.features[f])
                if miss is not None and miss < K - 1:
                    ll[miss] = np.clip(ll[miss] - np.median(ll[miss]), -self.max_missing_log_odds, self.max_missing_log_odds)
            self.tables_.append(ll)
        return self

    def log_joint(self, X):
        acc = np.zeros((X.shape[0], self.n_internal_))
        for f, ll in enumerate(self.tables_):
            acc += self.weights_[f] * ll[X[:, f]]
        return self.log_prior_ + self.lik_scale * acc

    def predict_proba(self, X):
        z = self.log_joint(X)
        z = z - z.max(1, keepdims=True)
        p = np.exp(z)
        return p / p.sum(1, keepdims=True)


class SubjectTokenNB:
    """Multinomial-style token evidence from the normalised subject (digits -> '#').
    Per-token log-likelihoods are centred across classes, clipped, and summed with 1/sqrt(n_known_tokens) scaling
    so long subjects cannot dominate. Unknown tokens and empty subjects contribute exactly 0."""

    def __init__(self, min_df=10, max_vocab=3000, alpha=0.5, clip=2.0):
        self.min_df, self.max_vocab, self.alpha, self.clip = int(min_df), int(max_vocab), float(alpha), float(clip)

    @staticmethod
    def _docs(texts):
        return [t.split() if isinstance(t, str) else [] for t in texts]

    def _matrix(self, docs):
        rows, cols = [], []
        for i, toks in enumerate(docs):
            for t in set(toks):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i); cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), len(self.vocab_)))

    def fit(self, texts, y, n_classes):
        docs = self._docs(texts)
        df_count = {}
        for d in docs:
            for t in set(d):
                df_count[t] = df_count.get(t, 0) + 1
        items = sorted(((t, c) for t, c in df_count.items() if c >= self.min_df), key=lambda x: (-x[1], x[0]))
        items = items[: self.max_vocab]
        self.vocab_ = {t: i for i, (t, _) in enumerate(items)}
        self.n_classes_ = n_classes
        V = len(self.vocab_)
        if V == 0:
            self.L_ = np.zeros((0, n_classes))
            return self
        X = self._matrix(docs)
        y = np.asarray(y, dtype=int)
        Y = sparse.csr_matrix((np.ones(len(y)), (y, np.arange(len(y)))), shape=(n_classes, len(y)))
        cnt = np.asarray((Y @ X).todense())
        tot = cnt.sum(1, keepdims=True)
        L = np.log((cnt + self.alpha) / (tot + self.alpha * V)).T
        L = L - L.mean(1, keepdims=True)
        self.L_ = np.clip(L, -self.clip, self.clip)
        return self

    def contribution(self, texts):
        n_docs = len(texts)
        if len(self.vocab_) == 0:
            return np.zeros((n_docs, self.n_classes_))
        X = self._matrix(self._docs(texts))
        n = np.asarray(X.sum(1)).ravel()
        return np.asarray(X @ self.L_) / np.sqrt(np.maximum(n, 1.0))[:, None]


# ----------------------------------------------------------------------------- calibration helpers
def _finalize(p):
    p = np.clip(p, PROB_FLOOR, 1.0 - PROB_FLOOR)
    return p / p.sum(1, keepdims=True)


def apply_temperature(P, T):
    z = np.log(np.clip(P, PROB_FLOOR, 1.0)) / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return _finalize(e / e.sum(1, keepdims=True))


def fit_temperature(P, y):
    from sklearn.metrics import log_loss
    res = minimize_scalar(lambda t: log_loss(y, apply_temperature(P, t), labels=[0, 1, 2]),
                          bounds=(0.2, 10.0), method="bounded")
    return float(res.x)


def fast_confusion_stats(y, pred):
    cm = np.bincount(np.asarray(y) * 3 + np.asarray(pred), minlength=9).reshape(3, 3).astype(float)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    prec = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    rec = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return f1, prec, rec


def offset_table(P, y, grid=np.linspace(-1.5, 3.5, 26)):
    """Evaluate every (o_spam, o_phish) pair of the grid for the rule argmax(log P + [0, o_spam, o_phish]).
    Only the labels `y` passed in are used - call it with INNER/TRAINING labels only."""
    L = np.log(np.clip(P, PROB_FLOOR, 1.0))
    y = np.asarray(y)
    offs, f1m, pp, rr = [], [], [], []
    for o1 in grid:
        for o2 in grid:
            o = np.array([0.0, o1, o2])
            f1, prec, rec = fast_confusion_stats(y, (L + o).argmax(1))
            offs.append(o); f1m.append(f1.mean()); pp.append(prec[2]); rr.append(rec[2])
    return dict(offsets=np.array(offs), macro_f1=np.array(f1m), phish_precision=np.array(pp), phish_recall=np.array(rr))


def pick_offsets(tab, mode="macro_f1", min_phish_precision=0.65):
    """Choose offsets from an offset_table().
    mode='argmax'              : no offsets
    mode='macro_f1'            : maximise macro-F1 (ties -> smaller offsets)
    mode='recall_at_precision' : maximise phishing recall subject to phishing precision >= min_phish_precision
                                 (falls back to macro_f1 and says so in info['feasible'] if no pair satisfies it)
    Returns (offsets, info)."""
    mag = np.abs(tab["offsets"]).sum(1)
    n = len(mag)
    if mode == "argmax":
        i = int(np.argmin(mag))
        return np.zeros(3), dict(mode_used="argmax", feasible=True, macro_f1=float("nan"),
                                 phish_precision=float("nan"), phish_recall=float("nan"))
    def best(ix, key):
        return max(ix, key=key)
    i_f1 = best(range(n), lambda i: (round(tab["macro_f1"][i], 9), -mag[i]))
    chosen, used, feas = i_f1, "macro_f1", True
    if mode == "recall_at_precision":
        ok = [i for i in range(n) if tab["phish_precision"][i] >= min_phish_precision]
        if ok:
            chosen = best(ok, lambda i: (round(tab["phish_recall"][i], 9), round(tab["macro_f1"][i], 9), -mag[i]))
            used = "recall_at_precision"
        else:
            feas = False
    return tab["offsets"][chosen].copy(), dict(mode_used=used, feasible=feas, macro_f1=float(tab["macro_f1"][chosen]),
                                               phish_precision=float(tab["phish_precision"][chosen]),
                                               phish_recall=float(tab["phish_recall"][chosen]))


def tune_class_offsets(P, y, grid=np.linspace(-1.5, 3.5, 26), mode="macro_f1", min_phish_precision=0.65):
    """Log-offsets [0, o_spam, o_phish] for argmax(log P + o). Probabilities are NOT altered.
    Returns (offsets, macro_f1_at_offsets). Thin wrapper kept for backwards compatibility."""
    o, info = pick_offsets(offset_table(P, y, grid), mode, min_phish_precision)
    return o, info["macro_f1"]


# ----------------------------------------------------------------------------- agent
class HeaderAgent:
    def __init__(self, features=None, n_bins=24, alpha=1.0, lik_scale=1.0, internal_4class=False,
                 max_missing_log_odds=1.5, feat_bound=2.5, use_missingness_features=True,
                 group_weighting="flat", subject_tokens=False, token_weight=1.0, smoothing="marginal"):
        raw = list(features) if features is not None else ALL_FEATURES
        self.use_missingness_features = bool(use_missingness_features)
        self.features = raw if self.use_missingness_features else [f for f in raw if f not in MISSINGNESS_FEATURES]
        self.n_bins, self.alpha, self.lik_scale = int(n_bins), float(alpha), float(lik_scale)
        self.internal_4class = bool(internal_4class)
        self.max_missing_log_odds, self.feat_bound = float(max_missing_log_odds), float(feat_bound)
        self.group_weighting, self.subject_tokens = group_weighting, bool(subject_tokens)
        self.token_weight, self.smoothing = float(token_weight), smoothing
        self.temperature = 1.0
        self.class_offsets = np.zeros(3)
        self.classes_ = np.array([0, 1, 2])
        self.is_fitted_ = False

    def _weights(self):
        if self.group_weighting != "inv_group":
            return np.ones(len(self.features))
        sizes = {g: sum(f in self.features for f in fs) for g, fs in FEATURE_GROUPS.items()}
        mean_n = np.mean([n for n in sizes.values() if n > 0])
        w = []
        for f in self.features:
            g = next(g for g, fs in FEATURE_GROUPS.items() if f in fs)
            w.append(np.sqrt(mean_n / sizes[g]))
        return np.array(w)

    def fit_features(self, Xf, y):
        y = np.asarray(y, dtype=int)
        C = 4 if self.internal_4class else 3
        self.preprocessor_ = HeaderPreprocessor(self.features, self.n_bins).fit(Xf)
        Z = self.preprocessor_.transform(Xf)
        raw = np.bincount(y, minlength=C).astype(float)
        self.training_prior_ = (np.array([raw[0], raw[1], raw[2] + raw[3]]) if C == 4 else raw[:3]) / raw.sum()
        self.model_ = SmoothedCategoricalNB(
            alpha=self.alpha, lik_scale=self.lik_scale, internal_4class=self.internal_4class,
            max_missing_log_odds=self.max_missing_log_odds, feat_bound=self.feat_bound,
            smoothing=self.smoothing).fit(Z, y, self.preprocessor_.n_categories_, self.preprocessor_, self._weights())
        self.token_nb_ = None
        if self.subject_tokens:
            self.token_nb_ = SubjectTokenNB().fit(Xf["u_norm"].tolist(), y, C)
        self.temperature, self.class_offsets = 1.0, np.zeros(3)
        self.is_fitted_ = True
        return self

    def set_temperature(self, T):
        self.temperature = float(T)

    def set_class_offsets(self, o):
        self.class_offsets = np.asarray(o, dtype=float)

    def proba_features(self, Xf, calibrated=True):
        if not self.is_fitted_:
            raise RuntimeError("Model is not fitted.")
        lj = self.model_.log_joint(self.preprocessor_.transform(Xf))
        if self.token_nb_ is not None:
            lj = lj + self.token_weight * self.token_nb_.contribution(Xf["u_norm"].tolist())
        lj = lj - lj.max(1, keepdims=True)
        p = np.exp(lj)
        p = p / p.sum(1, keepdims=True)
        if self.internal_4class:
            p = np.column_stack([p[:, 0], p[:, 1], p[:, 2] + p[:, 3]])
        p = _finalize(p)
        return apply_temperature(p, self.temperature) if calibrated else p

    def predict_label_features(self, Xf):
        P = self.proba_features(Xf, calibrated=True)
        return (np.log(P) + self.class_offsets).argmax(1)

    @staticmethod
    def completeness(Xf):
        s = Xf["s_missing"].to_numpy().astype(int)
        r = Xf["r_missing"].to_numpy().astype(int)
        d = (Xf["d_date_status"] == 0).to_numpy().astype(int)
        u = Xf["u_missing"].to_numpy().astype(int)
        return ((1 - s) + (1 - r) + (1 - d) + (1 - u)) / 4.0

    def detailed_features(self, Xf, index=None):
        P = self.proba_features(Xf, calibrated=True)
        comp = self.completeness(Xf)
        return pd.DataFrame({
            "p_safe": P[:, 0], "p_spam": P[:, 1], "p_phishing": P[:, 2],
            "nb_score": P[:, 1] + P[:, 2],
            "pred_class": (np.log(P) + self.class_offsets).argmax(1),
            "header_completeness": comp, "insufficient_evidence": (comp <= 0.5).astype(int)},
            index=Xf.index if index is None else index)

    # ---- raw-header convenience API
    def fit(self, X_emails, y):
        return self.fit_features(extract_features(_ensure_df(X_emails)), y)

    def predict_proba(self, X_emails, calibrated=True):
        df = _ensure_df(X_emails)
        return pd.DataFrame(self.proba_features(extract_features(df), calibrated), index=df.index, columns=list(CLASS_NAMES))

    def predict_detailed(self, X_emails):
        df = _ensure_df(X_emails)
        return self.detailed_features(extract_features(df), df.index)

    def save(self, path):
        import joblib
        joblib.dump({"agent": self, "version": __version__}, path)

    @classmethod
    def load(cls, path):
        import joblib
        return joblib.load(path)["agent"]


def score_emails(agent, emails):
    """One [0,1] non-safe score per email: P(spam)+P(phishing) (calibrated)."""
    p = agent.predict_proba(emails, calibrated=True).to_numpy(dtype=float)
    return np.clip(p[:, 1] + p[:, 2], 0.0, 1.0)


def score_email(agent, email):
    return float(score_emails(agent, [email])[0])
'''

import importlib
MODULE_PATH = OUT / "agent1_header.py"
MODULE_PATH.write_text(HEADER_SRC, encoding="utf-8")
sys.path.insert(0, str(OUT.resolve()))
os.environ["PYTHONPATH"] = str(OUT.resolve()) + os.pathsep + os.environ.get("PYTHONPATH", "")   # joblib workers import the same module
sys.modules.pop("agent1_header", None)
import agent1_header as A1
CLASS_NAMES = list(A1.CLASS_NAMES)
assert CLASS_NAMES == ["safe", "spam", "phishing"]
print("Agent 1 module", A1.__version__, "| class order", CLASS_NAMES)

Agent 1 module 7.0.0 | class order ['safe', 'spam', 'phishing']


## 2. Unit Testing and Parser Robustness Verification

Before accessing the dataset, this suite executes focused unit tests to mathematically verify parser integrity, feature boundary conditions, and numerical stability:
* **Targeted Missingness Bounding:** Confirms that missing sender, receiver, date, or subject fields are properly regularized and do not produce pathological near-100% phishing confidence on otherwise benign emails.
* **Strict Date and Timezone Handling:** Verifies that unparseable or missing date headers trigger feature abstention rather than defaulting to UTC, while fractional timezones (e.g., UTC+05:30) are correctly converted to hours.
* **Spoofing Heuristics:** Tests boundary-aware brand matching and lookalike domain detection.
* **Parity with Scikit-Learn:** Confirms mathematical convergence with `sklearn.naive_bayes.CategoricalNB` under unregularized baseline settings.
* **Inference Invariance:** Verifies that single-email dictionary scoring yields identical outputs to batch DataFrame processing.

In [7]:
import numpy as np, pandas as pd
from sklearn.naive_bayes import CategoricalNB


def run_unit_tests(A):
    ok = []
    def check(name, cond):
        assert cond, f"FAILED: {name}"
        ok.append(name)

    base = {"sender": "a@b.com", "receiver": "x@y.com", "date": "Mon, 12 Oct 2026 14:00:00 +0000", "subject": "hi"}

    # B2: weekday is real. 12 Oct 2026 = Monday(0); 16 Oct = Friday(4); 18 Oct = Sunday(6)
    F = A.extract_features(pd.DataFrame([
        {**base}, {**base, "date": "Fri, 16 Oct 2026 10:00:00 -0500"}, {**base, "date": "Sun, 18 Oct 2026 23:00:00 GMT"}]))
    check("B2 weekday computed (0,4,6)", F["d_weekday"].tolist() == [0, 4, 6])
    check("B2 tz parsed (0,-5,0)", F["d_tz_h"].tolist() == [0, -5, 0])
    F = A.extract_features(pd.DataFrame([{**base, "date": "Thu, 30 Feb 2026 10:00:00 +0000"}]))
    check("B3 impossible calendar date -> status 3", F["d_date_status"].iloc[0] == 3)

    # B3: unparseable / tz-less dates abstain for the derived date features
    rng = np.random.default_rng(0)
    N = 400
    rows = [{**base, "sender": f"u{i}@d{i%9}.com",
             "date": ["Mon, 12 Oct 2026 14:00:00 +0000", "garbage", "", "Tue, 13 Oct 2026 10:00:00"][i % 4]} for i in range(N)]
    df = pd.DataFrame(rows); y = rng.integers(0, 3, N)
    ag = A.HeaderAgent(n_bins=8).fit(df, y)
    Z = ag.preprocessor_.transform(A.extract_features(df.iloc[:4]))
    names, K = ag.preprocessor_.features, ag.preprocessor_.n_categories_
    ab = lambda f, r: Z[r, names.index(f)] == K[names.index(f)] - 1
    check("B3 garbage date: hour/weekday/tz abstain", ab("d_hour", 1) and ab("d_weekday", 1) and ab("d_tz_h", 1))
    check("B3 missing date: hour/weekday/tz abstain", ab("d_hour", 2) and ab("d_weekday", 2) and ab("d_tz_h", 2))
    check("B3 date without tz: hour kept, tz abstains", (not ab("d_hour", 3)) and ab("d_tz_h", 3))
    check("B3 valid date: nothing abstains", not (ab("d_hour", 0) or ab("d_weekday", 0) or ab("d_tz_h", 0)))

    # B4: undisclosed-recipients survives
    d4 = pd.DataFrame([{**base, "receiver": "undisclosed-recipients:;"}, {**base, "receiver": None}, {**base}])
    F4 = A.extract_features(d4)
    check("B4 undisclosed != missing", F4["r_missing"].tolist() == [0, 1, 0] and F4["r_has_undisclosed"].tolist() == [1, 0, 0])
    Z4 = ag.preprocessor_.transform(F4)
    j = names.index("r_has_undisclosed")
    check("B4 undisclosed feature is NOT abstained", Z4[0, j] == 1 and Z4[1, j] == K[j] - 1)

    # R1: compact receiver state
    F4s = A.extract_features(pd.DataFrame([{**base, "receiver": r} for r in
        [None, "undisclosed-recipients:;", "x@y.com", "a@b.com, c@d.com", "Undisclosed <u@b.com>"]]))
    check("R1 r_state = missing/undisclosed/single/several/undisclosed+addr", F4s["r_state"].tolist() == [0, 1, 2, 3, 1])
    check("R1 compact feature sets", set(A.COMPACT_RECEIVER) <= set(A.ALL_FEATURES_COMPACT) and "r_state" not in A.ALL_FEATURES
          and len([f for f in A.ALL_FEATURES_COMPACT if f.startswith("r_")]) == 3)

    # B5: brand logic
    doms = {"us.com": 0, "its.com": 0, "up.org": 0, "city.com": 0, "apply.com": 0, "dell.com": 0,
            "paypa1.com": 1, "paypall.com": 1, "goggle.com": 1, "paypal.com": 0, "ups.com": 0, "irs.gov": 0}
    F5 = A.extract_features(pd.DataFrame([{**base, "sender": f"a@{d}"} for d in doms]))
    check("B5 typo-brand: no short-token / common-word false positives, real typos caught",
          F5["s_typo_brand"].tolist() == list(doms.values()))
    F5 = A.extract_features(pd.DataFrame([{**base, "sender": s} for s in
                                          ["a@paypal.com.evil.ru", "a@secure.paypal.evil.com", "a@mail.paypal.com", "a@paypal.com"]]))
    check("B5 sub-domain brand spoof flagged only when registrable domain differs", F5["s_brand_spoof_host"].tolist() == [1, 1, 0, 0])
    F5 = A.extract_features(pd.DataFrame([{**base, "sender": s} for s in ["a@192.168.4.7", "a@[10.0.0.1]", "a@xn--pypal-4ve.com"]]))
    check("B5 IPv4 / punycode senders", F5["s_ip_domain"].tolist() == [1, 1, 0] and F5["s_punycode"].tolist() == [0, 0, 1]
          and F5["s_dom"].iloc[0] == "192.168.4.7")

    # Likelihood maths ------------------------------------------------------------------------
    rng = np.random.default_rng(3)
    n = 50000; yy = rng.choice(3, n, p=[.4, .5, .1])
    X = np.column_stack([(rng.random(n) < np.array([.2, .6, .9])[yy]).astype(int),
                         rng.integers(0, 5, n), rng.integers(0, 4, n)])
    Kc = [3 + 1, 5 + 1, 4 + 1]           # real categories + 1 abstain code (never used here)
    # parity with sklearn CategoricalNB when smoothing='laplace', no clipping, no abstain use
    ours = A.SmoothedCategoricalNB(alpha=1.0, feat_bound=0, max_missing_log_odds=0, smoothing="laplace").fit(X, yy, Kc)
    sk = CategoricalNB(alpha=1.0, min_categories=[3, 5, 4]).fit(X, yy)
    check("parity with sklearn.CategoricalNB (laplace, no abstain)", np.allclose(ours.predict_proba(X), sk.predict_proba(X), atol=1e-9))

    # N1: conditional on non-abstain, rows of P(cat|class) sum to 1 over real categories
    Xa = X.copy(); Ka = [4, 6, 5]
    mask = rng.random(n) < np.array([.05, .5, .8])[yy]      # abstain rate depends on class
    Xa[mask, 1] = Ka[1] - 1
    m = A.SmoothedCategoricalNB(alpha=1.0, feat_bound=0, max_missing_log_odds=0).fit(Xa, yy, Ka)
    P = np.exp(m.tables_[1][: Ka[1] - 1]).T                 # (class, real category)
    check("N1 P(cat|class, not abstained) sums to 1 per class", np.allclose(P.sum(1), 1.0))
    # feature 1 is independent of class given non-abstain -> its likelihood must be ~equal across classes
    spread = np.abs(m.tables_[1][: Ka[1] - 1] - m.tables_[1][: Ka[1] - 1].mean(1, keepdims=True)).max()
    check("N1 class-dependent abstain rate no longer leaks into the likelihood", spread < 0.15)
    check("abstain code contributes exactly 0", np.all(m.tables_[1][Ka[1] - 1] == 0))
    lj = np.log(np.bincount(yy) / n) + sum(t[Xa[:, f]] for f, t in enumerate(m.tables_))
    check("log_joint equals manual computation", np.allclose(lj, m.log_joint(Xa)))

    # Missing-category cap still enforced end to end
    ag2 = A.HeaderAgent(n_bins=8, max_missing_log_odds=1.0).fit(df, y)
    for jj, f in enumerate(ag2.features):
        mi = ag2.preprocessor_.missing_category(f)
        if mi is not None and mi < ag2.preprocessor_.n_categories_[jj] - 1:
            row = ag2.model_.tables_[jj][mi]
            assert np.abs(row - np.median(row)).max() <= 1.0 + 1e-8, f
    check("missing-category cap", True)

    # Subject-token block
    tx = ["verify your account now"] * 60 + ["team meeting notes"] * 60 + ["verify # offer"] * 20
    ty = np.array([2] * 60 + [0] * 60 + [1] * 20)
    tk = A.SubjectTokenNB(min_df=5).fit(tx, ty, 3)
    c = tk.contribution(["verify account", "zzz qqq", "", None])
    check("token block: phishing words push class 2, unknown/empty/None contribute 0",
          c[0].argmax() == 2 and np.allclose(c[1:], 0))

    # Weights
    w = A.HeaderAgent(group_weighting="inv_group")._weights()
    check("inv_group weights positive & finite", np.isfinite(w).all() and (w > 0).all())

    # Robustness to hostile inputs; probabilities always valid
    odd = [{}, {"sender": None}, {"sender": float("nan"), "subject": 5, "date": 1.5, "receiver": b"x@y.com"},
           {"sender": "no-at-sign", "receiver": "<>", "date": "\x00\x01", "subject": "\u202e" * 50},
           {"sender": "=?utf-8?b?4pyTIFRlc3Q=?= <A@B.CO.UK>", "receiver": "a@b.com, c@d.com", "subject": "RE: RE: Fwd: x" * 100},
           {"sender": "\"a@b.com\" <evil@c.ru>", "receiver": "undisclosed-recipients:;", "date": "Mon, 99 Foo 2026 99:99:99"},
           {"sender": "x" * 20000 + "@y.com", "subject": "a" * 100000}]
    ag3 = A.HeaderAgent(n_bins=8, subject_tokens=True).fit(df.assign(subject="verify your account"), y)
    p = ag3.predict_proba(odd).to_numpy()
    check("hostile/odd inputs: finite, in [0,1], rows sum to 1",
          np.isfinite(p).all() and (p >= 0).all() and np.allclose(p.sum(1), 1.0))
    check("class order", A.CLASS_NAMES == ("safe", "spam", "phishing") and list(ag3.predict_proba(odd).columns) == list(A.CLASS_NAMES))
    check("single == batch scoring", abs(A.score_email(ag3, odd[2]) - A.score_emails(ag3, odd)[2]) < 1e-12)

    # Calibration helpers
    P0 = ag3.proba_features(A.extract_features(df), calibrated=False)
    T = A.fit_temperature(P0, y)
    Pc = A.apply_temperature(P0, T)
    check("temperature keeps rows normalised", np.allclose(Pc.sum(1), 1.0) and 0.2 <= T <= 10)
    o, f = A.tune_class_offsets(P0, y, grid=np.linspace(-1, 1, 5))
    check("offsets: safe fixed at 0, shape 3", o.shape == (3,) and o[0] == 0)
    rr = np.random.default_rng(5); ys = rr.choice(3, 3000, p=[.45, .45, .10])
    lg = rr.normal(size=(3000, 3)) + 1.2 * np.eye(3)[ys] * np.array([1, 1, 1.0]); Ps = np.exp(lg - lg.max(1, keepdims=True)); Ps /= Ps.sum(1, keepdims=True)
    from sklearn.metrics import f1_score as _f1
    o_f1, _ = A.tune_class_offsets(Ps, ys, grid=np.linspace(-1, 3, 17))
    o_rp, _ = A.tune_class_offsets(Ps, ys, grid=np.linspace(-1, 3, 17), mode="recall_at_precision", min_phish_precision=0.5)
    Ls = np.log(np.clip(Ps, 1e-6, 1))
    f1a, pra, rca = A.fast_confusion_stats(ys, (Ls + o_f1).argmax(1)); f1b, prb, rcb = A.fast_confusion_stats(ys, (Ls + o_rp).argmax(1))
    check("offset tuner: fast macro-F1 == sklearn", abs(f1a.mean() - _f1(ys, (Ls + o_f1).argmax(1), average="macro")) < 1e-9)
    check("offset tuner: recall_at_precision honours the precision floor and does not lose recall", prb[2] >= 0.5 and rcb[2] >= rca[2] - 1e-9)
    check("determinism", np.allclose(P0, ag3.proba_features(A.extract_features(df), calibrated=False)))

    # ---- v7 regression tests ------------------------------------------------------------------
    # Q1: quantile edges must come from rows that carry evidence (missing subjects have u_len == 0 as a placeholder)
    n2 = 600
    d2 = pd.DataFrame({"sender": [f"u{i}@d{i % 7}.com" for i in range(n2)], "receiver": "a@b.com",
                       "date": "Mon, 12 Oct 2026 14:00:00 +0000",
                       "subject": ["" if i % 2 == 0 else "x" * (20 + i % 40) for i in range(n2)]})
    F2 = A.extract_features(d2)
    pp2 = A.HeaderPreprocessor(["u_len", "u_missing"]).fit(F2)
    check("Q1 u_len bin edges ignore missing subjects", min(pp2.edges_["u_len"]) >= 20)
    # Q2: receiver-same-domain has no meaning without a sender
    F3 = A.extract_features(pd.DataFrame([{**base, "sender": None}, {**base}]))
    pp3 = A.HeaderPreprocessor(["r_same_domain_as_sender", "r_is_freemail"]).fit(F3)
    Z3 = pp3.transform(F3)
    check("Q2 r_same_domain_as_sender abstains when the sender is missing",
          Z3[0, 0] == pp3.n_categories_[0] - 1 and Z3[1, 0] != pp3.n_categories_[0] - 1)
    # Q3: decision-rule picker
    tab = A.offset_table(Ps, ys, grid=np.linspace(-1, 3, 17))
    o_a, _ = A.pick_offsets(tab, "argmax")
    check("argmax rule has zero offsets", np.all(o_a == 0))
    o_x, i_x = A.pick_offsets(tab, "recall_at_precision", min_phish_precision=1.01)
    check("infeasible precision floor falls back to macro-F1 and says so", (not i_x["feasible"]) and i_x["mode_used"] == "macro_f1")
    o_y, i_y = A.pick_offsets(tab, "recall_at_precision", min_phish_precision=0.5)
    check("feasible precision floor is honoured", i_y["feasible"] and i_y["phish_precision"] >= 0.5)
    # single-email vs batch parity on the *full* detailed output (probabilities + decision + completeness)
    det_b = ag3.predict_detailed(odd); det_1 = pd.concat([ag3.predict_detailed([o]) for o in odd], ignore_index=True)
    check("single == batch for predict_detailed", np.allclose(det_b.reset_index(drop=True).to_numpy(float), det_1.to_numpy(float)))
    return ok


passed = run_unit_tests(A1)
print(f"{len(passed)} unit checks passed")

36 unit checks passed


## 3. Dataset Loading, Target Formulation, and Input Integrity

### Dataset Specifications
The benchmark dataset (`emails_clean.csv`) contains 48,939 email records partitioned into three fixed splits:
* **Training Set:** 34,256 records
* **Validation Set:** 7,340 records
* **Testing Set:** 7,343 records

### Operational Label Formulation
The dataset aggregates records from four historical email corpora: `CEAS_08`, `SpamAssassin`, `Nazario`, and `Nigerian_Fraud`. Target classes are mapped according to the documented operational convention:
* **Safe (`0`):** Legitimate emails across all source corpora (original label `0`).
* **Spam (`1`):** Unsolicited commercial email from `CEAS_08` and `SpamAssassin` (original label `1`).
* **Phishing (`2`):** Malicious emails from `Nazario` (credential phishing) and `Nigerian_Fraud` (advance-fee fraud).

> **Methodological Note:** These target labels represent an operational convention inherited from historical benchmark corpora. They should not be treated as independently verified ground truth for modern production email.

### Input Hygiene and Quarantine
* Empty strings are exclusively recognized as missing headers (`keep_default_na=False`), ensuring valid textual subjects such as `"None"` or `"NA"` are preserved.
* **Strict Label Isolation:** Validation and test set labels are quarantined at this stage and are not loaded or inspected.

In [8]:
CSV_PATH = None            # <- set to a local path if you do not use $EMAILS_CSV / Kaggle
searched = [CSV_PATH, os.environ.get("EMAILS_CSV", ""), "emails_clean.csv"]
searched += glob.glob("/kaggle/input/**/emails_clean.csv", recursive=True) + glob.glob("**/emails_clean.csv", recursive=True)
DATA_PATH = next((os.path.abspath(c) for c in searched if c and os.path.isfile(c)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "emails_clean.csv was not found.\n  -> Kaggle: add the dataset that contains it to the notebook (Add Input).\n"
        "  -> Local : put the file next to this notebook, or set CSV_PATH above, or export EMAILS_CSV=/path/to/emails_clean.csv.\n"
        "Nothing was trained and no data was generated.")
print("Dataset:", DATA_PATH)
df = pd.read_csv(DATA_PATH, keep_default_na=False, na_values=[""])
required = ["email_id", "split", "source", "sender", "receiver", "date", "subject", "label"]
miss = [c for c in required if c not in df.columns]
if miss: raise ValueError(f"missing columns: {miss} (found: {list(df.columns)})")
if df["email_id"].isna().any() or not df["email_id"].is_unique: raise ValueError("email_id must be unique and non-null")
if df["label"].isna().any() or df["source"].isna().any(): raise ValueError("label and source must not be empty")
aliases = {"train": "train", "training": "train", "val": "val", "validation": "val", "valid": "val", "test": "test", "testing": "test"}
df["split"] = df["split"].astype(str).str.strip().str.lower().map(aliases)
if df["split"].isna().any() or set(df["split"]) != {"train", "val", "test"}:
    raise ValueError("split must contain exactly train / val / test (aliases: training, validation, testing)")
train_df, val_df, test_df = [df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")]

SPAM_SOURCES, PHISH_SOURCES = {"ceas08", "spamassasin", "spamassassin"}, {"nazario", "nigerianfraud"}
norm_source = lambda x: "".join(ch for ch in str(x).lower() if ch.isalnum())

def map_labels(frame):
    """-> (3-class target 0 safe / 1 spam / 2 phishing, internal 4-class target: 0 safe, 1 spam, 2 Nazario, 3 Nigerian_Fraud).
    Raw 0 = safe. Raw 1 is mapped from the corpus definition. Unknown sources / labels raise (no invented classes)."""
    raw = pd.to_numeric(frame["label"], errors="raise").astype(int).to_numpy()
    if not set(np.unique(raw)) <= {0, 1}: raise ValueError(f"unexpected raw labels {np.unique(raw)} (expected 0/1)")
    y3, y4 = np.zeros(len(frame), int), np.zeros(len(frame), int)
    for i, (r, s) in enumerate(zip(raw, frame["source"].map(norm_source))):
        if s not in SPAM_SOURCES | PHISH_SOURCES: raise ValueError(f"unknown source {s!r}: add it to SPAM_SOURCES or PHISH_SOURCES explicitly")
        if r == 0: continue
        if s in SPAM_SOURCES: y3[i] = y4[i] = 1
        elif s == "nazario": y3[i], y4[i] = 2, 2
        else: y3[i], y4[i] = 2, 3
    return y3, y4

# sources are checked for ALL splits (no labels read for val/test)
for nm, fr in (("train", train_df), ("val", val_df), ("test", test_df)):
    bad = set(fr["source"].map(norm_source)) - (SPAM_SOURCES | PHISH_SOURCES)
    if bad: raise ValueError(f"{nm}: unknown sources {bad}")
y_train, y_train_internal = map_labels(train_df)
src_train = train_df["source"].map(norm_source).to_numpy()
if set(np.unique(y_train)) != {0, 1, 2}: raise ValueError("training split lacks a class")
print("Rows:", dict(train=len(train_df), val=len(val_df), test=len(test_df)))
print("Training class counts [safe, spam, phishing]:", np.bincount(y_train).tolist(), "| prior", (np.bincount(y_train) / len(y_train)).round(3).tolist())
print("\nTRAIN: raw label x source -> mapped class (phishing corpora should have NO raw-0 rows if they contain only phishing):")
print(pd.crosstab([train_df["source"], train_df["label"].astype(int)], y_train, rownames=["source", "raw_label"], colnames=["class"]))
print("\nVAL / TEST rows per source (labels intentionally not inspected here):")
print(pd.crosstab(df["source"], df["split"]))

Dataset: /kaggle/input/datasets/itsduaa/cleanone/emails_clean.csv
Rows: {'train': 34256, 'val': 7340, 'test': 7343}
Training class counts [safe, spam, phishing]: [14952, 15987, 3317] | prior [0.436, 0.467, 0.097]

TRAIN: raw label x source -> mapped class (phishing corpora should have NO raw-0 rows if they contain only phishing):
class                         0      1     2
source         raw_label                    
CEAS_08        0          12136      0     0
               1              0  14813     0
Nazario        1              0      0  1062
Nigerian_Fraud 1              0      0  2255
SpamAssasin    0           2816      0     0
               1              0   1174     0

VAL / TEST rows per source (labels intentionally not inspected here):
split           test  train   val
source                           
CEAS_08         5654  26949  5756
Nazario          236   1062   228
Nigerian_Fraud   525   2255   490
SpamAssasin      928   3990   866


## 4. Feature Extraction and Cross-Split Overlap Diagnostics

This section extracts the 36 engineered header features across all splits and performs an audit of inter-split overlap:
* **Header-Only Restriction:** Features are computed exclusively from `sender`, `receiver`, `date`, and `subject`. Corpus name (`source`), `email_id`, and `split` are excluded.
* **Overlap Measurement:** Quantifies the proportion of validation and test records that share exact sender addresses, registrable domains, or normalized subject templates with the training partition.
* **Diagnostic Motivation:** If a large fraction of holdout records share subject templates or sender domains with the training data, a standard random cross-validation split will suffer from substantial data leakage and produce over-optimistic performance estimates. This audit motivates the grouped validation strategies deployed in subsequent sections.

In [9]:
X_train = A1.extract_features(train_df[HEADER_COLS])
X_val   = A1.extract_features(val_df[HEADER_COLS])
X_test  = A1.extract_features(test_df[HEADER_COLS])
print("model features:", len(A1.ALL_FEATURES))

def group_keys(Xf, ids, level):
    out = []
    for addr, dom, i in zip(Xf["s_addr"], Xf["s_dom"], ids):
        if level == "identity": g = A1.canonical_sender_identity(dom, addr)   # domain, or address for free-mail/IP
        else: g = f"addr:{addr}" if addr else None
        out.append(g if g else f"missing:{i}")                                # a missing sender is its own singleton group
    return np.asarray(out, dtype=object)

G_ID   = group_keys(X_train, train_df["email_id"], "identity")
G_ADDR = group_keys(X_train, train_df["email_id"], "address")
templates = X_train["u_norm"].to_numpy()
vc = pd.Series(G_ID).value_counts()
print(f"\nSender groups (identity level): {len(vc):,} | largest group = {vc.iloc[0]:,} rows ({vc.iloc[0]/len(G_ID):.1%}) -> {vc.index[0]}")

# --- overlap of val/test with train (sender, template, near-duplicate)
def keysets(Xf, ids):
    return dict(addr=set(group_keys(Xf, ids, "address")), ident=set(group_keys(Xf, ids, "identity")),
                tpl={t for t in Xf["u_norm"] if t}, dup=set(zip(Xf["s_addr"], Xf["u_norm"])))
tr_k = keysets(X_train, train_df["email_id"]); rows = []
for name, Xf, fr in (("val", X_val, val_df), ("test", X_test, test_df)):
    a, i = group_keys(Xf, fr["email_id"], "address"), group_keys(Xf, fr["email_id"], "identity")
    rows.append({"split": name,
        "sender_address_seen_in_train": np.mean([k in tr_k["addr"] for k in a]),
        "sender_identity_seen_in_train": np.mean([k in tr_k["ident"] for k in i]),
        "subject_template_seen": np.mean([t in tr_k["tpl"] for t in Xf["u_norm"] if t]),
        "near_duplicate_sender+subject_seen": np.mean([k in tr_k["dup"] for k in zip(Xf["s_addr"], Xf["u_norm"])])})
print("\nShare of eval rows whose key also occurs in TRAIN (high = optimistic random-split estimate):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

# --- within-train near-duplicates and label conflicts
dk = pd.Series(list(zip(X_train["s_addr"], X_train["u_norm"])))
dupmask = dk.duplicated(keep=False).to_numpy() & (X_train["s_addr"].to_numpy() != "")
conf = pd.DataFrame({"k": dk, "y": y_train}).groupby("k")["y"].nunique()
print(f"\nTrain rows sharing (sender, subject-template) with another row: {dupmask.mean():.1%}; keys with conflicting labels: {(conf > 1).sum()}")
# exact duplicates of the four raw header strings
rawdup = train_df.duplicated(HEADER_COLS, keep=False)
print(f"Train rows whose four raw header strings are identical to another row: {rawdup.mean():.1%}")

# --- corpus signature
sig = X_train.assign(source=train_df["source"].values)
sig["date_missing"] = (sig.d_date_status == 0); sig["date_unparseable"] = (sig.d_date_status == 3)
sig["recv_state"] = sig.r_state
tab = sig.groupby("source")[["s_missing", "r_missing", "u_missing", "date_missing", "date_unparseable",
                              "r_has_undisclosed", "s_is_freemail", "s_ip_domain", "s_suspicious_tld", "s_has_num"]].mean().round(3)
tab["top_tz"] = sig[sig.d_date_status == 2].groupby("source")["d_tz_h"].agg(lambda s: s.value_counts().head(3).to_dict())
tab["top_tld"] = sig.groupby("source")["s_tld"].agg(lambda s: s.value_counts().head(3).to_dict())
print("\nPer-source header habits (large differences = corpus fingerprints):"); print(tab.to_string())
print("\nReceiver format per source (r_state: 0 missing, 1 undisclosed, 2 single, 3 several), row-normalised:")
print(pd.crosstab(sig.source, sig.recv_state, normalize="index").round(3).to_string())
print("\nSender-domain concentration per source (share of rows from the top-5 registrable domains):")
print(pd.DataFrame({s: [g["s_dom"].value_counts(normalize=True).head(5).sum(), g["s_dom"].nunique()] for s, g in sig.groupby("source")},
                   index=["top5_share", "n_distinct_domains"]).T.round(3).to_string())
print("\nMissing-sender rows by class (a perfect class indicator would be a pure shortcut):")
print(pd.crosstab(X_train.s_missing.map({0: "sender present", 1: "sender missing"}), pd.Series(y_train).map(dict(enumerate(CLASS_NAMES)))).to_string())

model features: 36

Sender groups (identity level): 15,865 | largest group = 463 rows (1.4%) -> dom:apache.org

Share of eval rows whose key also occurs in TRAIN (high = optimistic random-split estimate):
split  sender_address_seen_in_train  sender_identity_seen_in_train  subject_template_seen  near_duplicate_sender+subject_seen
  val                         0.428                          0.637                  0.670                               0.207
 test                         0.436                          0.637                  0.663                               0.204

Train rows sharing (sender, subject-template) with another row: 18.9%; keys with conflicting labels: 2
Train rows whose four raw header strings are identical to another row: 0.1%

Per-source header habits (large differences = corpus fingerprints):
                s_missing  r_missing  u_missing  date_missing  date_unparseable  r_has_undisclosed  s_is_freemail  s_ip_domain  s_suspicious_tld  s_has_num             

## 5. Grouped Cross-Validation and Data Partitioning

To obtain generalizable performance estimates, training data must be partitioned such that information cannot leak between folds:
* **Sender Identity Grouping:** Groups are defined by canonical sender identity (registrable domain for custom domains; exact email address for free-mail providers and IP hosts). Emails from the same domain or address are strictly confined to a single fold.
* **StratifiedGroupKFold:** Partitions the 34,256 training records into 5 folds, preserving multi-class label proportions while guaranteeing zero sender overlap between training and validation folds.
* **Template-Disjoint Partitioning Helper:** Provides a secondary partitioning function (`make_group_folds(..., purge=True)`) that identifies emails sharing subject templates with the evaluation split and purges them from the training fold, enabling isolated generalization audits.

In [10]:
n_train = len(y_train)
ALL_IDX = np.arange(n_train)
assert len(X_train) == n_train == len(train_df) == len(G_ID) == len(templates)
assert (X_train.index.to_numpy() == ALL_IDX).all(), "feature rows must align with train_df rows"

def _tpl_set(ix):
    return {t for t in templates[ix] if t}

def make_group_folds(idx, groups, n_splits, seed, purge=False):
    """Stratified group folds over the subset `idx`. Returns [(train_idx, eval_idx)] as absolute positions.
    Always asserts: groups never straddle train/eval, every class survives in the training part.
    purge=True additionally drops training rows whose subject template occurs in the evaluation part and asserts template-disjointness."""
    idx = np.asarray(idx)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for k, (tr_rel, va_rel) in enumerate(cv.split(np.zeros(len(idx)), y_train[idx], groups=groups[idx])):
        tr, va = idx[tr_rel], idx[va_rel]
        if set(groups[tr]) & set(groups[va]): raise AssertionError(f"group leakage in fold {k}")
        if purge:
            bad = _tpl_set(va)
            tr = tr[np.array([(not t) or (t not in bad) for t in templates[tr]], dtype=bool)]
            if _tpl_set(tr) & _tpl_set(va): raise AssertionError(f"template leakage in fold {k}")
        if len(np.unique(y_train[tr])) != 3: raise ValueError(f"fold {k}: training part lacks a class")
        if len(va) == 0: raise ValueError(f"fold {k}: empty evaluation part")
        out.append((tr, va))
    cov = np.sort(np.concatenate([va for _, va in out]))
    if not np.array_equal(cov, np.sort(idx)): raise AssertionError("evaluation parts must partition idx exactly once")
    return out

# ---------------- final grouped CV (also the OUTER folds of the nested experiments)
try:
    GROUPS, GROUP_LEVEL = G_ID, "identity (domain / free-mail address)"
    folds = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED)
except ValueError as e:
    print("!! identity-level grouping unusable:", e, "-> falling back to address-level groups")
    GROUPS, GROUP_LEVEL = G_ADDR, "address"
    folds = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED)
print("CV grouping:", GROUP_LEVEL)
FOLD_ID = np.full(n_train, -1)
for k, (tr, va) in enumerate(folds):
    FOLD_ID[va] = k
    tr_t, v = _tpl_set(tr), [t for t in templates[va] if t]
    print(f"fold {k}: train={len(tr):,} holdout={len(va):,} classes={np.bincount(y_train[va], minlength=3).tolist()} seen-template={np.mean([t in tr_t for t in v]):.1%}")
assert (FOLD_ID >= 0).all()

# ---------------- feature sets and candidate grids
_nc = lambda fl: [f for f in fl if f not in A1.CLOCK_FEATURES]
FEATURE_SETS = {"full": list(A1.ALL_FEATURES), "no_clock": _nc(A1.ALL_FEATURES),
                "compact_recv": list(A1.ALL_FEATURES_COMPACT), "compact_no_clock": _nc(A1.ALL_FEATURES_COMPACT)}

def build_grid(kind):
    """full: the 96-candidate grid | reduced: 32 candidates with lik_scale fixed at 1.0 (the PREVIOUS methodology; kept only for the old-vs-new comparison, because
    fixing lik_scale used knowledge from a full-dataset search) | stress: 24 candidates (4 feature sets x subject tokens x lik_scale; flat weights, 3-class NB)
    used by the template / LOSO stress tests. No candidate list is ever derived from outer-fold results."""
    if kind == "full":      lik, gw, tok, i4 = ((1.0, 2.0) if QUICK_SEARCH else (1.0, 1.5, 2.0)), (("flat",) if QUICK_SEARCH else ("flat", "inv_group")), (False, True), (False, True)
    elif kind == "reduced": lik, gw, tok, i4 = (1.0,), (("flat",) if QUICK_SEARCH else ("flat", "inv_group")), (False, True), (False, True)
    elif kind == "stress":  lik, gw, tok, i4 = (1.0, 1.5, 2.0), ("flat",), (False, True), (False,)
    else: raise ValueError(kind)
    return [dict(feature_set=fs, n_bins=16, alpha=1.0, lik_scale=l, group_weighting=g, subject_tokens=t, internal_4class=i,
                 feat_bound=2.5, max_missing_log_odds=1.0)
            for fs in FEATURE_SETS for l in lik for g in gw for t in tok for i in i4]

FULL_GRID, NESTED_GRID, STRESS_GRID = build_grid("full"), build_grid(NESTED_GRID_MODE), build_grid("stress")
REDUCED_GRID = build_grid("reduced")
if not QUICK_SEARCH:
    assert NESTED_GRID_MODE == "full" and len(NESTED_GRID) == 96 == len(FULL_GRID), "the nested search must use all 96 configurations (no silent fallback to a reduced grid)"
print(f"candidate grids: full={len(FULL_GRID)} nested({NESTED_GRID_MODE})={len(NESTED_GRID)} reduced(old, comparison only)={len(REDUCED_GRID)} stress={len(STRESS_GRID)}")
complexity = lambda c: len(FEATURE_SETS[c["feature_set"]]) // 8 + c["subject_tokens"] * 2 + c["internal_4class"] + (c["group_weighting"] != "flat")

def cfg_to_agent_kwargs(c):
    kw = {k: v for k, v in c.items() if k != "feature_set"}
    kw["features"] = FEATURE_SETS[c["feature_set"]]
    return kw

# ---------------- fitting (parallel when N_JOBS > 1)
_UNUSED = ["s_addr", "s_dom"]          # string columns the model never reads: dropped before shipping data to workers (faster, same result)
def _slim(Xf, cfg):
    drop = _UNUSED + ([] if cfg["subject_tokens"] else ["u_norm"])
    return Xf.drop(columns=[c for c in drop if c in Xf.columns])

def _fit_proba(kw, Xtr, ytr, Xva):
    import agent1_header as A
    m = A.HeaderAgent(**kw).fit_features(Xtr, ytr)
    return m.proba_features(Xva, calibrated=False)

def run_tasks(tasks):
    if N_JOBS <= 1 or len(tasks) < 2: return [_fit_proba(*t) for t in tasks]
    return Parallel(n_jobs=N_JOBS, backend="loky")(delayed(_fit_proba)(*t) for t in tasks)

# ---------------- metrics
lg = lambda P: np.log(np.clip(P, 1e-6, 1))
def predict_from_P(P, offsets=None):
    return (lg(P) + (0 if offsets is None else offsets)).argmax(1)

def safe_ap(ytrue_bool, score):
    ytrue_bool = np.asarray(ytrue_bool)
    return float(average_precision_score(ytrue_bool, score)) if 0 < ytrue_bool.sum() < len(ytrue_bool) else float("nan")

def ece_top(P, y, bins=10):
    conf, hit = P.max(1), (P.argmax(1) == y)
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(conf[m].mean() - hit[m].mean())
    return e

def metrics(y, P, offsets=None, pred=None):
    """Full metric dict. Phishing precision/recall are NaN when undefined (no predicted / no true phishing rows); AP is NaN when the slice has only one class."""
    y = np.asarray(y); pred = predict_from_P(P, offsets) if pred is None else np.asarray(pred)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    present = [c for c in range(3) if sup[c] > 0]
    nonph = y != 2
    return dict(n=len(y), n_safe=int(sup[0]), n_spam=int(sup[1]), n_phish=int(sup[2]), accuracy=accuracy_score(y, pred),
                macro_f1=float(np.mean(f1[present])) if present else float("nan"),
                phish_precision=pr[2] if (pred == 2).any() else float("nan"), phish_recall=rc[2] if sup[2] > 0 else float("nan"),
                phish_f1=f1[2] if sup[2] > 0 else float("nan"),
                phish_fpr=float((pred[nonph] == 2).mean()) if nonph.any() else float("nan"),
                phish_AP=safe_ap(y == 2, P[:, 2]), nonsafe_AP=safe_ap(y != 0, 1 - P[:, 0]),
                safe_recall=rc[0] if sup[0] > 0 else float("nan"), spam_recall=rc[1] if sup[1] > 0 else float("nan"),
                log_loss=float(log_loss(y, P, labels=[0, 1, 2])), ece=ece_top(P, y))

def full_report(y, P, offsets, title, pred=None):
    pred = predict_from_P(P, offsets) if pred is None else pred
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    print(f"\n=== {title} ===")
    print(pd.DataFrame({"precision": pr, "recall": rc, "f1": f1, "support": sup}, index=CLASS_NAMES).round(4).to_string())
    print("confusion (rows=true, cols=pred):")
    print(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1, 2]), index=[f"true_{c}" for c in CLASS_NAMES], columns=[f"pred_{c}" for c in CLASS_NAMES]).to_string())
    print({k: round(float(v), 4) for k, v in metrics(y, P, offsets, pred).items() if k not in ("n", "n_safe", "n_spam", "n_phish")})

def by_source(y, P, pred, sources):
    rows = []
    for s in np.unique(sources):
        m = sources == s; r = dict(source=s, n=int(m.sum()))
        for c, nm in enumerate(CLASS_NAMES):
            mc = m & (y == c)
            if mc.any(): r[f"recall_{nm}"] = round(float((pred[mc] == c).mean()), 4)
        r["nonsafe_recall"] = round(float((pred[m & (y != 0)] != 0).mean()), 4) if (m & (y != 0)).any() else np.nan
        r["false_phishing_rate_on_nonphish"] = round(float((pred[m & (y != 2)] == 2).mean()), 4) if (m & (y != 2)).any() else np.nan
        rows.append(r)
    return pd.DataFrame(rows)

def decision_name(mode=None, floor=None):
    mode = OFFSET_MODE if mode is None else mode
    return {"argmax": "argmax", "macro_f1": "macro_f1"}.get(mode, f"recall@P>={(MIN_PHISH_PRECISION if floor is None else floor):.2f}")
print("helpers ready | decision rule for exports:", decision_name())

CV grouping: identity (domain / free-mail address)
fold 0: train=27,214 holdout=7,042 classes=[2896, 3186, 960] seen-template=57.2%
fold 1: train=28,176 holdout=6,080 classes=[2355, 3058, 667] seen-template=60.4%
fold 2: train=27,204 holdout=7,052 classes=[3367, 3099, 586] seen-template=58.7%
fold 3: train=27,423 holdout=6,833 classes=[2815, 3457, 561] seen-template=58.5%
fold 4: train=27,007 holdout=7,249 classes=[3519, 3187, 543] seen-template=57.0%
candidate grids: full=96 nested(full)=96 reduced(old, comparison only)=32 stress=24
helpers ready | decision rule for exports: macro_f1


## 6. Diagnostic: Feature-Group Source Identifiability Audit

A central risk in multi-corpus email datasets is that a machine learning model may learn to recognize the *origin corpus* rather than generalizable *spam or phishing behavior*.

This diagnostic evaluates this risk by attempting to predict the **source corpus** from isolated feature groups using 5-fold sender-grouped cross-validation:
* **4-Source Balanced Accuracy (Chance = 0.25):** Measures overall corpus separability from headers alone.
* **CEAS_08 vs. SpamAssassin Accuracy (Chance = 0.50):** Both corpora contain both safe and spam email. High separability between them represents a pure corpus fingerprint that provides no genuine signal for spam detection.

Feature groups exhibiting disproportionate source identifiability are flagged as potential shortcut candidates and analyzed in the ablation experiments (Section 11).

In [11]:
from sklearn.naive_bayes import CategoricalNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import balanced_accuracy_score

src_codes, src_names = pd.factorize(src_train)
GROUP_DEFS = {
    "sender (all s_*)": A1.FEATURE_GROUPS["sender"], "receiver (all r_*)": A1.FEATURE_GROUPS["receiver"],
    "date (status+clock)": A1.FEATURE_GROUPS["date"], "clock only (hour/weekday/tz)": list(A1.CLOCK_FEATURES),
    "missingness flags only": [f for f in A1.MISSINGNESS_FEATURES if f in A1.SPEC], "subject shape (u_*)": A1.FEATURE_GROUPS["subject"],
    "all model features (full)": FEATURE_SETS["full"]}

def _src_cv(feats, rows, target):
    pred = np.full(n_train, -1)
    sub = np.asarray(rows)
    for tr, va in StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(sub)), target[sub], groups=GROUPS[sub]):
        tr, va = sub[tr], sub[va]
        if feats == "tokens":
            cv = CountVectorizer(binary=True, min_df=10, max_features=3000, token_pattern=r"\S+")
            Ztr = cv.fit_transform(templates[tr]); Zva = cv.transform(templates[va]); nb = MultinomialNB(alpha=0.5)
        else:
            pp = A1.HeaderPreprocessor(feats).fit(X_train.iloc[tr])
            Ztr, Zva = pp.transform(X_train.iloc[tr]), pp.transform(X_train.iloc[va])
            nb = CategoricalNB(alpha=1.0, min_categories=pp.n_categories_)
        pred[va] = nb.fit(Ztr, target[tr]).predict(Zva)
    return balanced_accuracy_score(target[sub], pred[sub])

ceas_sa = np.where(np.isin(src_train, ["ceas08", "spamassasin", "spamassassin"]))[0]
cs_target = (src_train == "ceas08").astype(int)
rows = []
t0 = time.time()
for name, feats in list(GROUP_DEFS.items()) + [("subject tokens (words)", "tokens")]:
    rows.append({"feature group": name, "4-source bal.acc (chance .25)": _src_cv(feats, ALL_IDX, src_codes),
                 "CEAS vs SpamAssassin bal.acc (chance .50)": _src_cv(feats, ceas_sa, cs_target)})
SRC_ID = pd.DataFrame(rows).set_index("feature group")
print(SRC_ID.round(3).to_string()); print(f"({time.time()-t0:.0f}s)")
print("\nHow to read: values well above chance mean that group can recognise the corpus. Those groups are candidate shortcuts; the ablations in sections 11-12 test whether they\n"
      "still help on data from a different sender set / template set / corpus before anyone drops them.")

                              4-source bal.acc (chance .25)  CEAS vs SpamAssassin bal.acc (chance .50)
feature group                                                                                         
sender (all s_*)                                      0.290                                      0.524
receiver (all r_*)                                    0.360                                      0.526
date (status+clock)                                   0.375                                      0.676
clock only (hour/weekday/tz)                          0.372                                      0.677
missingness flags only                                0.382                                      0.525
subject shape (u_*)                                   0.435                                      0.514
all model features (full)                             0.556                                      0.703
subject tokens (words)                                0.751              

## 7. Nested Cross-Validation and Hyperparameter Optimization

### Why Nested Cross-Validation is Necessary
Evaluating hyperparameter tuning, probability calibration, and decision threshold selection on the same cross-validation folds used to report performance introduces selection bias (*optimism bias*). Nested Cross-Validation resolves this by strictly decoupling model selection from model evaluation:
* **Outer Loop (5 Sender-Disjoint Folds):** Used exclusively to estimate generalization performance on unseen sender identities.
* **Inner Loop (5 Inner Folds):** Executed entirely within each outer training partition to select the optimal model configuration, fit the calibration temperature, and determine decision offsets.

### Candidate Search Space (96 Factorial Configurations)
Inside every outer fold, the inner loop explores a full grid:
1. **Likelihood Scaling (`lik_scale`):** $0.5, 0.8, 1.0$ (tempering feature likelihood contributions).
2. **Feature Group Weighting:** `uniform` vs. `informative` (attenuating noisy feature groups).
3. **Feature Subsets:** `full`, `no_missingness`, `reduced_anti_corpus`, and `structure_only`.
4. **Subject Token Model:** Unigram token Naive Bayes enabled vs. disabled.
5. **Class Formulation:** Standard 3-class model vs. internal 4-class model (separating Nazario and Nigerian Fraud internally before collapsing probabilities to 3 classes).

The best configuration is selected via the **One-Standard-Error (1-SE) rule** on Macro-F1. The outer fold is scored strictly once using the chosen configuration.

In [12]:
SEL_GRID = np.linspace(-1.0, 3.0, 9)            # coarse offset grid, used only to rank candidates
FLOORS = sorted(set(PRECISION_FLOORS) | {MIN_PHISH_PRECISION})
OP_NAMES = ["argmax", "macro_f1"] + [f"recall@P>={f:.2f}" for f in FLOORS]
CFG_KEYS = list(NESTED_GRID[0].keys())

def operating_points(P_cal, y):
    """All decision rules, tuned on (P_cal, y) only. Returns {name: (offsets, info)}."""
    tab = A1.offset_table(P_cal, y)
    ops = {"argmax": A1.pick_offsets(tab, "argmax"), "macro_f1": A1.pick_offsets(tab, "macro_f1")}
    for f in FLOORS: ops[f"recall@P>={f:.2f}"] = A1.pick_offsets(tab, "recall_at_precision", f)
    return ops

def search_and_calibrate(idx, groups, grid, seed, purge=False, n_inner=N_INNER, ifolds=None, forbid=None):
    """Everything that is *chosen* is chosen here, from the rows `idx` ONLY (labels y_train[idx]):
       inner grouped folds -> candidate scores -> one-SE choice -> temperature.  Returns the chosen config, T and the inner OOF probabilities.
       Decision offsets are then tuned by operating_points(sel['P_cal'], y_train[idx]).
       `forbid` = rows that must play NO role (the outer evaluation rows): asserted disjoint from every row used here."""
    idx = np.asarray(idx)
    if forbid is not None: assert not (set(idx.tolist()) & set(np.asarray(forbid).tolist())), "outer-evaluation rows reached the inner search"
    ifolds = make_group_folds(idx, groups, n_inner, seed, purge) if ifolds is None else ifolds
    if forbid is not None:
        used = np.concatenate([np.concatenate([tr, va]) for tr, va in ifolds]); assert not (set(used.tolist()) & set(np.asarray(forbid).tolist()))
    four_ok = all(set(np.unique(y_train_internal[tr])) == {0, 1, 2, 3} for tr, _ in ifolds)
    cands = [c for c in grid if four_ok or not c["internal_4class"]]
    tasks = []
    for cfg in cands:
        kw = cfg_to_agent_kwargs(cfg); yf = y_train_internal if cfg["internal_4class"] else y_train
        for tr, va in ifolds: tasks.append((kw, _slim(X_train.iloc[tr], cfg), yf[tr], _slim(X_train.iloc[va], cfg)))
    res = iter(run_tasks(tasks))
    rel = np.full(n_train, -1); rel[idx] = np.arange(len(idx))
    y_in = y_train[idx]; rows, Ps = [], []
    for ci, cfg in enumerate(cands):
        P = np.zeros((len(idx), 3))
        for tr, va in ifolds: P[rel[va]] = next(res)
        Ps.append(P)
        o_c, _ = A1.tune_class_offsets(P, y_in, grid=SEL_GRID)                 # offsets tuned on INNER pooled OOF only
        per = []
        for tr, va in ifolds:
            r = rel[va]; f1k, _, _ = A1.fast_confusion_stats(y_train[va], (lg(P[r]) + o_c).argmax(1))
            per.append((f1k.mean(), safe_ap(y_train[va] == 2, P[r, 2]), safe_ap(y_train[va] != 0, 1 - P[r, 0])))
        pf = np.array(per); S = np.nanmean(pf, axis=1); ok = np.isfinite(S)
        f1a, pra, rca = A1.fast_confusion_stats(y_in, (lg(P) + o_c).argmax(1))
        rows.append({**cfg, "cand": ci, "complexity": complexity(cfg), "score": S[ok].mean(),
                     "score_se": (S[ok].std(ddof=1) / np.sqrt(ok.sum())) if ok.sum() > 1 else 0.0,
                     "macro_f1_tuned": np.nanmean(pf[:, 0]), "phish_AP": np.nanmean(pf[:, 1]), "nonsafe_AP": np.nanmean(pf[:, 2]),
                     "log_loss": log_loss(y_in, P, labels=[0, 1, 2]), "phish_P@tuned": pra[2], "phish_R@tuned": rca[2]})
    R = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    best = R.iloc[0]
    eligible = R[R.score >= best.score - best.score_se].sort_values(["complexity", "score"], ascending=[True, False])
    chosen = eligible.iloc[0]; ci = int(chosen["cand"])
    T = A1.fit_temperature(Ps[ci], y_in)
    return dict(cfg=cands[ci], T=T, P_raw=Ps[ci], P_cal=A1.apply_temperature(Ps[ci], T), idx=idx, ifolds=ifolds,
                table=R, n_eligible=len(eligible), n_cands=len(cands))

def fit_one(cfg, tr):
    yf = y_train_internal if cfg["internal_4class"] else y_train
    return A1.HeaderAgent(**cfg_to_agent_kwargs(cfg)).fit_features(X_train.iloc[tr], yf[tr])

DATA_SHA = hashlib.sha256(pathlib.Path(DATA_PATH).read_bytes()).hexdigest() if "DATA_PATH" in globals() else ""
MODULE_SHA = hashlib.sha256(HEADER_SRC.encode("utf-8")).hexdigest()

def _cache_path(label, k, tr, grid, seed, purge, keep_models):
    key = json.dumps([label, k, int(seed), bool(purge), bool(keep_models), len(tr), hashlib.sha256(np.asarray(tr, dtype=np.int64).tobytes()).hexdigest(),
                      [sorted(c.items()) for c in grid], DATA_SHA, MODULE_SHA, N_INNER, FLOORS, list(map(float, SEL_GRID))], default=str)
    return OUT / "cache" / (hashlib.sha256(key.encode()).hexdigest()[:24] + ".joblib")

def run_nested(outer_folds, groups, grid, purge=False, label="", keep_models=False, seed_base=1000):
    """Outer loop. Per outer fold: search_and_calibrate + operating points on the OUTER TRAINING rows only (assertion: outer-evaluation rows are forbidden),
    then ONE fit on the outer training rows and prediction of the untouched outer evaluation rows.
    Returned: stacked outer predictions P / preds (per decision rule) AND the per-fold temperature and offsets (fold_T / fold_ops), so every row can be
    traced to the exact procedure that produced it. Finished folds are checkpointed (USE_CACHE)."""
    t0 = time.time()
    P_out = np.zeros((n_train, 3)); covered = np.zeros(n_train, bool); fold_of = np.full(n_train, -1)
    preds = {n: np.full(n_train, -1) for n in OP_NAMES}; log, models, fold_T, fold_ops = [], [], [], []
    for k, (tr, va) in enumerate(outer_folds):
        assert not (set(tr) & set(va)), "outer train/eval overlap"
        cp = _cache_path(label, k, tr, grid, seed_base + k, purge, keep_models)
        if USE_CACHE and cp.exists():
            r = joblib.load(cp); sel_cfg, T_k, ops, Pcal, mdl, lrow = r["cfg"], r["T"], r["ops"], r["Pcal"], r["model"], r["log"]
            src = "cache"
        else:
            sel = search_and_calibrate(tr, groups, grid, seed=seed_base + k, purge=purge, forbid=va)
            ops = operating_points(sel["P_cal"], y_train[tr])                       # inner labels only
            mdl = fit_one(sel["cfg"], tr)
            Pcal = A1.apply_temperature(mdl.proba_features(X_train.iloc[va], calibrated=False), sel["T"])
            sel_cfg, T_k = sel["cfg"], sel["T"]
            lrow = {"fold": k, "n_train": len(tr), "n_eval": len(va), **{c: sel_cfg[c] for c in ("feature_set", "lik_scale", "group_weighting", "subject_tokens", "internal_4class")},
                    "n_cands": sel["n_cands"], "n_within_1SE": sel["n_eligible"], "T": round(T_k, 3),
                    "offsets_macro_f1": tuple(float(x) for x in ops["macro_f1"][0].round(2))}
            if USE_CACHE: joblib.dump(dict(cfg=sel_cfg, T=T_k, ops=ops, Pcal=Pcal, model=(mdl if keep_models else None), log=lrow), cp)
            src = "computed"
        P_out[va] = Pcal; covered[va] = True; fold_of[va] = k
        for n, (o, info) in ops.items(): preds[n][va] = predict_from_P(Pcal, o)
        log.append(lrow); fold_T.append(T_k); fold_ops.append(ops)
        if keep_models: models.append(dict(model=mdl, T=T_k, ops=ops, va=va, tr=tr))
        el = time.time() - t0
        print(f"  [{label}] outer fold {k + 1}/{len(outer_folds)} ({src}, {el:.0f}s elapsed, ETA {el / (k + 1) * (len(outer_folds) - k - 1):.0f}s) -> {sel_cfg['feature_set']}, "
              f"lik={sel_cfg['lik_scale']}, tokens={sel_cfg['subject_tokens']}, 4cls={sel_cfg['internal_4class']}", flush=True)
    assert covered.all(), "every row must be predicted by an outer fold exactly once"
    return dict(P=P_out, preds=preds, log=pd.DataFrame(log), models=models, label=label, folds=outer_folds, fold_T=fold_T, fold_ops=fold_ops, fold_of=fold_of)

def nested_table(res, y=None, rows_mask=None, names=None):
    """metrics per decision rule on the stacked outer predictions."""
    y = y_train if y is None else y; mk = np.ones(len(y), bool) if rows_mask is None else rows_mask
    out = {}
    for n in (OP_NAMES if names is None else names):
        out[n] = metrics(y[mk], res["P"][mk], pred=res["preds"][n][mk])
    return pd.DataFrame(out).T
print("nested machinery ready | decision rules:", OP_NAMES)

nested machinery ready | decision rules: ['argmax', 'macro_f1', 'recall@P>=0.60', 'recall@P>=0.65', 'recall@P>=0.70', 'recall@P>=0.75', 'recall@P>=0.80']


In [13]:
# ---- pipeline sanity tests (run on the real training data, before the expensive experiments)
# 1) no-peeking: scrambling the labels of the OUTER EVALUATION rows must not change anything chosen from the outer training rows
_tr, _va = folds[0]
_small = STRESS_GRID[:4]
def _choose():
    s = search_and_calibrate(_tr, GROUPS, _small, seed=SEED + 999)
    o = operating_points(s["P_cal"], y_train[_tr])
    return s["cfg"], s["T"], {n: tuple(v[0].round(6)) for n, v in o.items()}
ref = _choose()
y3_bak, y4_bak = y_train.copy(), y_train_internal.copy()
try:
    perm = np.random.default_rng(0).permutation(len(_va))
    y_train[_va] = y3_bak[_va][perm]; y_train_internal[_va] = y4_bak[_va][perm]
    alt = _choose()
finally:
    y_train[:] = y3_bak; y_train_internal[:] = y4_bak
assert ref[0] == alt[0] and abs(ref[1] - alt[1]) < 1e-12 and ref[2] == alt[2], "outer-fold labels influenced model selection / calibration / offsets!"
print("TEST 1 passed: model choice, temperature and all offsets are independent of the outer evaluation labels")

# 2) every fold structure is verified at construction; re-check the headline folds explicitly
for k, (tr, va) in enumerate(folds):
    assert not (set(GROUPS[tr]) & set(GROUPS[va])), k
    assert (FOLD_ID[va] == k).all()
print("TEST 2 passed: outer folds are sender-disjoint and FOLD_ID matches the folds")

# 3) metrics helper vs sklearn
_rng = np.random.default_rng(1); _y = _rng.integers(0, 3, 500); _P = _rng.dirichlet(np.ones(3), 500)
_m = metrics(_y, _P); assert abs(_m["macro_f1"] - f1_score(_y, _P.argmax(1), average="macro")) < 1e-12
assert abs(_m["phish_AP"] - average_precision_score(_y == 2, _P[:, 2])) < 1e-12
print("TEST 3 passed: metrics() agrees with scikit-learn")

# 4) no-peeking also holds when the held-out rows' templates are purged (the strict cross-fitting used for Agent 3 training predictions)
_fp = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED, purge=True)
_tr, _va = _fp[0]
def _choose_p():
    s = search_and_calibrate(_tr, GROUPS, _small, seed=SEED + 998, purge=True, forbid=_va)
    o = operating_points(s["P_cal"], y_train[_tr])
    return s["cfg"], s["T"], {n: tuple(v[0].round(6)) for n, v in o.items()}
ref_p = _choose_p()
try:
    perm = np.random.default_rng(1).permutation(len(_va))
    y_train[_va] = y3_bak[_va][perm]; y_train_internal[_va] = y4_bak[_va][perm]
    alt_p = _choose_p()
finally:
    y_train[:] = y3_bak; y_train_internal[:] = y4_bak
assert ref_p[0] == alt_p[0] and abs(ref_p[1] - alt_p[1]) < 1e-12 and ref_p[2] == alt_p[2], "purged cross-fitting is not independent of the held-out labels!"
# the search must refuse to touch forbidden rows
try:
    search_and_calibrate(ALL_IDX, GROUPS, _small, seed=1, forbid=_va); raise SystemExit("forbid guard did not fire")
except AssertionError: pass
print("TEST 4 passed: purged cross-fitting is independent of the held-out labels; the forbid-guard rejects outer rows")

# 5) shipping slimmed feature frames to workers does not change the model
_cfg = STRESS_GRID[3]; _tr0, _va0 = folds[0]
_a = _fit_proba(cfg_to_agent_kwargs(_cfg), X_train.iloc[_tr0], (y_train_internal if _cfg["internal_4class"] else y_train)[_tr0], X_train.iloc[_va0])
_b = _fit_proba(cfg_to_agent_kwargs(_cfg), _slim(X_train.iloc[_tr0], _cfg), (y_train_internal if _cfg["internal_4class"] else y_train)[_tr0], _slim(X_train.iloc[_va0], _cfg))
assert np.array_equal(_a, _b), "slimmed frames changed the probabilities"
print("TEST 5 passed: slimmed worker frames give bit-identical probabilities")

TEST 1 passed: model choice, temperature and all offsets are independent of the outer evaluation labels
TEST 2 passed: outer folds are sender-disjoint and FOLD_ID matches the folds
TEST 3 passed: metrics() agrees with scikit-learn
TEST 4 passed: purged cross-fitting is independent of the held-out labels; the forbid-guard rejects outer rows
TEST 5 passed: slimmed worker frames give bit-identical probabilities


In [14]:
t0 = time.time()
NEST_SENDER = run_nested(folds, GROUPS, NESTED_GRID, purge=False, label="sender-disjoint", keep_models=True, seed_base=1000)
print(f"\nnested sender-disjoint CV finished in {(time.time() - t0) / 60:.1f} min")
print("\nConfiguration chosen INSIDE each outer fold (stability of the selection):")
print(NEST_SENDER["log"].to_string(index=False))

DEC = decision_name()
print(f"\n##### NESTED sender-disjoint results, decision rule = {DEC} (each row predicted by a pipeline that never used it for any choice) #####")
full_report(y_train, NEST_SENDER["P"], None, f"Nested CV, sender-disjoint, rule={DEC}", pred=NEST_SENDER["preds"][DEC])
full_report(y_train, NEST_SENDER["P"], None, "Nested CV, sender-disjoint, plain argmax", pred=NEST_SENDER["preds"]["argmax"])
print("\nPer-outer-fold stability:")
print(pd.DataFrame([metrics(y_train[va], NEST_SENDER["P"][va], pred=NEST_SENDER["preds"][DEC][va]) for _, va in folds])
      [["macro_f1", "phish_precision", "phish_recall", "phish_AP", "log_loss"]].round(4).to_string())

  [sender-disjoint] outer fold 1/5 (computed, 50s elapsed, ETA 200s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender-disjoint] outer fold 2/5 (computed, 99s elapsed, ETA 148s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender-disjoint] outer fold 3/5 (computed, 146s elapsed, ETA 97s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender-disjoint] outer fold 4/5 (computed, 193s elapsed, ETA 48s) -> full, lik=1.0, tokens=True, 4cls=True
  [sender-disjoint] outer fold 5/5 (computed, 240s elapsed, ETA 0s) -> full, lik=1.0, tokens=True, 4cls=False

nested sender-disjoint CV finished in 4.0 min

Configuration chosen INSIDE each outer fold (stability of the selection):
 fold  n_train  n_eval feature_set  lik_scale group_weighting  subject_tokens  internal_4class  n_cands  n_within_1SE     T  offsets_macro_f1
    0    27214    7042        full        1.0            flat            True            False       96            12 1.617 (0.0, -0.1, -0.9)
    1    28176    6080        full       

## 8. Probability Calibration and Decision Operating-Point Tuning

### Separation of Probabilities and Decision Rules
This implementation decouples continuous probability estimation from discrete classification decisions:
1. **Calibrated Probabilities ($P(\text{safe}), P(\text{spam}), P(\text{phishing})$):** Calibrated via post-hoc temperature scaling ($T$), fitted to minimize multi-class log-loss on inner out-of-fold predictions. These probabilities preserve ranking order and are exported unaltered for Agent 3 stacking.
2. **Decision Operating Points (`pred_class`):** Discrete classification decisions are governed by class log-offsets added to the log-probabilities. Adjusting decision offsets shifts the operating trade-off between precision and recall without altering the underlying probability ranking (Phishing Average Precision remains invariant).

### Decision Operating Rules Evaluated:
* **`argmax`:** Standard maximum a posteriori classification.
* **`macro_f1`:** Selects class offsets that maximize balanced multi-class Macro-F1 on inner training predictions.
* **`recall@P>=x`:** Maximizes phishing recall subject to achieving a minimum phishing precision threshold ($P \ge 0.60, 0.70, 0.80$) on inner training predictions.

The resulting table reports realized nested precision, recall, and false-positive rates on unseen outer senders.

In [15]:
tab = nested_table(NEST_SENDER)
tab["target_floor"] = [np.nan, np.nan] + FLOORS
tab["precision_minus_floor"] = tab["phish_precision"] - tab["target_floor"]
print("NESTED operating points (sender-disjoint outer folds). Probabilities identical across rows; only the decision rule changes:")
print(tab[["phish_precision", "phish_recall", "phish_f1", "phish_fpr", "macro_f1", "accuracy", "safe_recall", "spam_recall", "target_floor", "precision_minus_floor"]].astype(float).round(4).to_string())
print("\nThreshold-free quality of the SAME probabilities (does not depend on the rule):",
      {k: round(float(v), 4) for k, v in metrics(y_train, NEST_SENDER["P"], pred=NEST_SENDER["preds"]["argmax"]).items() if k in ("phish_AP", "nonsafe_AP", "log_loss", "ece")})
prec_c, rec_c, _ = precision_recall_curve(y_train == 2, NEST_SENDER["P"][:, 2])
print("Best attainable phishing recall at a given precision, nested P (oracle threshold, upper bound):",
      {f">={p:.2f}": (round(float(rec_c[:-1][prec_c[:-1] >= p].max()), 3) if (prec_c[:-1] >= p).any() else None) for p in (0.8, 0.75, 0.7, 0.65, 0.6)})
# how often the precision floor was infeasible on inner data
print("\nOffsets chosen per outer fold (macro_f1 rule):", NEST_SENDER["log"]["offsets_macro_f1"].tolist())
print("\nWhere phishing goes, per phishing corpus (nested predictions; rows sum to 1):")
for nm, mm in (("Nazario", src_train == "nazario"), ("Nigerian_Fraud", src_train == "nigerianfraud")):
    for rule in ("argmax", DEC):
        pr_ = NEST_SENDER["preds"][rule][mm]
        print(f"  {nm:15s} {rule:15s} ->safe {np.mean(pr_ == 0):.3f}  ->spam {np.mean(pr_ == 1):.3f}  ->phishing {np.mean(pr_ == 2):.3f}")
print("\nPer-source results (nested, rule=%s). Phishing sources contain no safe mail: read recall_phishing with care." % DEC)
print(by_source(y_train, NEST_SENDER["P"], NEST_SENDER["preds"][DEC], src_train).to_string(index=False))
print("\nReliability (nested calibrated probabilities, top label):")
conf, hit = NEST_SENDER["P"].max(1), NEST_SENDER["P"].argmax(1) == y_train
bins = pd.cut(conf, [0, .4, .5, .6, .7, .8, .9, .95, 1.0])
print(pd.DataFrame({"n": pd.Series(hit).groupby(bins, observed=True).size(), "mean_conf": pd.Series(conf).groupby(bins, observed=True).mean(),
                    "accuracy": pd.Series(hit).groupby(bins, observed=True).mean()}).round(3).to_string())

NESTED operating points (sender-disjoint outer folds). Probabilities identical across rows; only the decision rule changes:
                phish_precision  phish_recall  phish_f1  phish_fpr  macro_f1  accuracy  safe_recall  spam_recall  target_floor  precision_minus_floor
argmax                   0.7334        0.8734    0.7973     0.0340    0.8890    0.9203       0.9240       0.9267           NaN                    NaN
macro_f1                 0.8089        0.8140    0.8114     0.0206    0.8951    0.9248       0.9403       0.9333           NaN                    NaN
recall@P>=0.60           0.6122        0.9237    0.7364     0.0627    0.8578    0.8967       0.8634       0.9221          0.60                 0.0122
recall@P>=0.65           0.6549        0.9120    0.7623     0.0515    0.8680    0.9027       0.8743       0.9273          0.65                 0.0049
recall@P>=0.70           0.7098        0.8842    0.7875     0.0388    0.8832    0.9155       0.9029       0.9339          0.70

## 9. Generalization Stress Tests: Subject-Template Dependence

While sender-disjoint cross-validation ensures that sender addresses do not leak across folds, many phishing and spam campaigns share normalized subject lines across distinct senders. 

To determine how heavily Agent 1 relies on recurring subject templates, this section evaluates four comparative protocols through the complete nested selection pipeline:
1. **Sender-Disjoint (Baseline):** Senders are strictly partitioned; templates may be shared across folds.
2. **Template-Disjoint:** Groups are defined by normalized subject templates; senders may overlap across folds.
3. **Sender-Disjoint + Template-Purged:** Sender-disjoint folds where all training emails sharing a subject template with the evaluation fold are explicitly removed from training.
4. **Jointly Disjoint Components:** Connected components of the bipartite sender-template graph held out as atomic units (executed if the largest component does not dominate the dataset).

Comparing these tiers reveals the generalization performance floor when encountering completely novel email templates.

In [16]:
TPL_RES = {}
# ---- groups for template-disjoint folds
G_TPL = np.array([("tpl:" + t) if t else f"emptytpl:{i}" for t, i in zip(templates, train_df["email_id"])], dtype=object)

# ---- connected components (sender group OR template links rows)
def joint_components(g_sender, g_tpl_raw):
    parent = {}
    def find(x):
        while parent.setdefault(x, x) != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    for i, (s, t) in enumerate(zip(g_sender, g_tpl_raw)):
        find(("s", s))
        if t: union(("s", s), ("t", t))
    return np.array([str(find(("s", s))) for s in g_sender], dtype=object)

G_JOINT = joint_components(GROUPS, templates)
comp_sizes = pd.Series(G_JOINT).value_counts()
largest_share = comp_sizes.iloc[0] / n_train
print(f"Joint sender/template components: {len(comp_sizes):,}; largest = {comp_sizes.iloc[0]:,} rows ({largest_share:.1%}); 2nd = {comp_sizes.iloc[1] if len(comp_sizes) > 1 else 0:,}")
JOINT_FEASIBLE = largest_share <= 1.5 / N_SPLITS * 0.5     # a component must fit well inside one fold (<= 15 % for 5 folds)

if RUN_TEMPLATE_EXPERIMENTS:
    t0 = time.time()
    f_tpl = make_group_folds(ALL_IDX, G_TPL, N_SPLITS, SEED + 1)
    for tr, va in f_tpl: assert not (_tpl_set(tr) & _tpl_set(va))                 # verified: template-disjoint
    TPL_RES["template-disjoint"] = run_nested(f_tpl, G_TPL, STRESS_GRID, label="template-disjoint", seed_base=2000)
    f_pur = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED, purge=True)
    for tr, va in f_pur:
        assert not (set(GROUPS[tr]) & set(GROUPS[va])) and not (_tpl_set(tr) & _tpl_set(va))   # verified: BOTH conditions
    print("purged training rows per fold:", [int(len(folds[k][0]) - len(f_pur[k][0])) for k in range(N_SPLITS)])
    TPL_RES["sender-disjoint + template-purged"] = run_nested(f_pur, GROUPS, STRESS_GRID, purge=True, label="sender+template purged", seed_base=3000)
    if JOINT_FEASIBLE:
        f_jt = make_group_folds(ALL_IDX, G_JOINT, N_SPLITS, SEED + 2)
        for tr, va in f_jt: assert not (set(GROUPS[tr]) & set(GROUPS[va])) and not (_tpl_set(tr) & _tpl_set(va))
        TPL_RES["jointly disjoint components"] = run_nested(f_jt, G_JOINT, STRESS_GRID, label="joint components", seed_base=4000)
    else:
        print(f"\nJointly-disjoint split NOT run: the largest sender/template component holds {largest_share:.0%} of the rows, so folds cannot be balanced or stratified.\n"
              "Reason: popular templates (e.g. list-server subjects) link many senders, and senders reuse many templates, so almost everything ends up in one connected component.\n"
              "The 'sender-disjoint + template-purged' experiment is the documented substitute: it is verified sender-disjoint AND template-disjoint, at the price of removing training rows.")
    print(f"\ntemplate experiments finished in {(time.time() - t0) / 60:.1f} min")

    # ---- comparison table
    cmp_rows = {"sender-disjoint (section 9, main)": metrics(y_train, NEST_SENDER["P"], pred=NEST_SENDER["preds"][DEC])}
    for k_, r_ in TPL_RES.items(): cmp_rows[k_] = metrics(y_train, r_["P"], pred=r_["preds"][DEC])
    CMP = pd.DataFrame(cmp_rows).T
    print(f"\nSender-disjoint vs template-disjoint evaluation (nested, rule={DEC}). NOTE: the grids differ (sender-disjoint: {len(NESTED_GRID)} candidates, stress tests: {len(STRESS_GRID)}).")
    print(CMP[["macro_f1", "accuracy", "phish_precision", "phish_recall", "phish_fpr", "phish_AP", "log_loss"]].astype(float).round(4).to_string())
    base_ = CMP.loc["sender-disjoint (section 9, main)"]
    print("\nDrop relative to the sender-disjoint estimate:")
    print((CMP[["macro_f1", "phish_recall", "phish_precision", "phish_AP"]].astype(float) - base_[["macro_f1", "phish_recall", "phish_precision", "phish_AP"]].astype(float)).round(4).to_string())
    print("\nChosen configurations (stress tests):")
    for k_, r_ in TPL_RES.items(): print(" ", k_, r_["log"][["feature_set", "subject_tokens"]].value_counts().to_dict())
else:
    print("RUN_TEMPLATE_EXPERIMENTS is False: skipped.")

Joint sender/template components: 6,545; largest = 23,354 rows (68.2%); 2nd = 170
  [template-disjoint] outer fold 1/5 (computed, 15s elapsed, ETA 61s) -> full, lik=1.0, tokens=True, 4cls=False
  [template-disjoint] outer fold 2/5 (computed, 31s elapsed, ETA 46s) -> full, lik=1.0, tokens=True, 4cls=False
  [template-disjoint] outer fold 3/5 (computed, 46s elapsed, ETA 30s) -> full, lik=1.0, tokens=True, 4cls=False
  [template-disjoint] outer fold 4/5 (computed, 61s elapsed, ETA 15s) -> full, lik=1.0, tokens=True, 4cls=False
  [template-disjoint] outer fold 5/5 (computed, 75s elapsed, ETA 0s) -> full, lik=1.0, tokens=True, 4cls=False
purged training rows per fold: [11473, 11248, 11221, 11161, 11627]
  [sender+template purged] outer fold 1/5 (computed, 9s elapsed, ETA 37s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender+template purged] outer fold 2/5 (computed, 19s elapsed, ETA 28s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender+template purged] outer fold 3/5 (computed, 28s el

In [17]:
# ---- 11B. Robustness inside the nested sender-disjoint predictions
print("B. Seen vs unseen subject templates, measured on the NESTED sender-disjoint predictions (template seen in that outer fold's training rows or not):")
seen = np.zeros(n_train, bool)
for k, (tr, va) in enumerate(folds):
    trt = _tpl_set(tr); seen[va] = [t in trt for t in templates[va]]
rows = {}
for nm, mk in (("seen template", seen), ("unseen or empty template", ~seen)):
    rows[nm] = metrics(y_train[mk], NEST_SENDER["P"][mk], pred=NEST_SENDER["preds"][DEC][mk])
print(pd.DataFrame(rows).T[["n", "accuracy", "macro_f1", "phish_precision", "phish_recall", "phish_AP", "log_loss", "ece"]].astype(float).round(4).to_string())

print("\nBy sender type (nested): free-mail vs other vs missing sender")
st = np.where(X_train.s_missing == 1, "missing_sender", np.where(X_train.s_is_freemail == 1, "freemail", "other_domain"))
print(pd.DataFrame({s: metrics(y_train[st == s], NEST_SENDER["P"][st == s], pred=NEST_SENDER["preds"][DEC][st == s]) for s in np.unique(st)}).T
      [["n", "accuracy", "macro_f1", "phish_precision", "phish_recall", "safe_recall", "spam_recall", "log_loss"]].astype(float).round(4).to_string())

# ---- field-removal scenarios, using the stored OUTER models (each applied only to its own untouched outer fold)
def scenario(drop=(), keep_only=None):
    cols = [c for c in HEADER_COLS if (c in keep_only if keep_only else c not in drop)]
    frame = train_df[HEADER_COLS].copy()
    for c in HEADER_COLS:
        if c not in cols: frame[c] = None
    Xs = A1.extract_features(frame); P = np.zeros((n_train, 3)); pred = np.zeros(n_train, int)
    for m in NEST_SENDER["models"]:
        Pk = A1.apply_temperature(m["model"].proba_features(Xs.iloc[m["va"]], calibrated=False), m["T"])
        P[m["va"]] = Pk; pred[m["va"]] = predict_from_P(Pk, m["ops"][DEC][0])
    return P, pred
SCEN = {"baseline (all fields)": {}, "no sender": dict(drop=("sender",)), "no receiver": dict(drop=("receiver",)), "no date": dict(drop=("date",)),
        "no subject": dict(drop=("subject",)), "sender only": dict(keep_only=("sender",)), "subject only": dict(keep_only=("subject",)),
        "date+receiver only": dict(keep_only=("date", "receiver"))}
rob = {}
for name, kw in SCEN.items():
    P, pr_ = scenario(**kw); rob[name] = {**metrics(y_train, P, pred=pr_), "mean_p_safe": P[:, 0].mean(), "mean_p_phish": P[:, 2].mean()}
print(f"\nC. Field-removal scenarios (nested outer models, rule={DEC}):")
print(pd.DataFrame(rob).T[["macro_f1", "accuracy", "phish_recall", "phish_precision", "safe_recall", "log_loss", "mean_p_safe", "mean_p_phish"]].astype(float).round(4).to_string())

B. Seen vs unseen subject templates, measured on the NESTED sender-disjoint predictions (template seen in that outer fold's training rows or not):
                                n  accuracy  macro_f1  phish_precision  phish_recall  phish_AP  log_loss     ece
seen template             19680.0    0.9795    0.9565           0.8793        0.9278    0.9638    0.0758  0.0145
unseen or empty template  14576.0    0.8510    0.8340           0.7790        0.7688    0.8539    0.4265  0.0190

By sender type (nested): free-mail vs other vs missing sender
                      n  accuracy  macro_f1  phish_precision  phish_recall  safe_recall  spam_recall  log_loss
freemail         3294.0    0.8940    0.8100           0.8361        0.9172       0.9671       0.4967    0.3155
missing_sender    632.0    0.9620    0.9632           0.9954        0.9046          NaN       0.9974    0.1015
other_domain    30330.0    0.9274    0.8889           0.7873        0.7857       0.9354       0.9449    0.2178

C. Fie

## 10. Out-of-Distribution Diagnostics: Leave-One-Source-Out (LOSO)

To assess robustness against distribution shift across distinct email collections, this experiment implements a **Leave-One-Source-Out (LOSO)** evaluation:
1. For each source corpus (`CEAS_08`, `SpamAssassin`, `Nazario`, `Nigerian_Fraud`), the entire corpus is excluded from training.
2. The complete inner optimization pipeline (candidate selection, temperature calibration, and decision offset tuning) is executed on the remaining three corpora.
3. The resulting model is evaluated on the held-out corpus.

> **Interpretation Note:** In this dataset, target labels are correlated with specific historical collections (all phishing originates from Nazario and Nigerian Fraud). Consequently, holding out a phishing corpus tests sensitivity to cross-corpus stylistic and temporal shift rather than pure feature degradation.

In [18]:
def run_loso(grid, label, seed_base=5000, verbose=True):
    t0 = time.time(); rows, cms = [], {}
    P_all = np.zeros((n_train, 3)); preds = {n: np.full(n_train, -1) for n in OP_NAMES}; covered = np.zeros(n_train, bool); chosen = {}
    for j, s in enumerate(sorted(np.unique(src_train))):
        tr, va = np.where(src_train != s)[0], np.where(src_train == s)[0]
        assert not (set(src_train[tr]) & {s}), "held-out source leaked into training"
        if len(np.unique(y_train[tr])) < 3:
            rows.append(dict(held_out=s, note="remaining corpora lack a class")); continue
        try:
            sel = search_and_calibrate(tr, GROUPS, grid, seed=seed_base + j)       # selection + T from remaining corpora only
        except ValueError as e:
            rows.append(dict(held_out=s, note=f"inner CV impossible: {e}")); continue
        ops = operating_points(sel["P_cal"], y_train[tr])                            # offsets from remaining corpora only
        m = fit_one(sel["cfg"], tr)
        P = A1.apply_temperature(m.proba_features(X_train.iloc[va], calibrated=False), sel["T"])
        P_all[va] = P; covered[va] = True; chosen[s] = (sel["cfg"]["feature_set"], sel["cfg"]["subject_tokens"], round(sel["T"], 2))
        for n, (o, info) in ops.items(): preds[n][va] = predict_from_P(P, o)
        for rule in dict.fromkeys(["argmax", DEC]):
            r = dict(held_out=s, rule=rule, **metrics(y_train[va], P, pred=preds[rule][va]), config=chosen[s][0], tokens=chosen[s][1], T=chosen[s][2])
            rows.append(r)
            cms[(s, rule)] = confusion_matrix(y_train[va], preds[rule][va], labels=[0, 1, 2])
    if verbose: print(f"  [{label}] LOSO finished in {(time.time() - t0) / 60:.1f} min")
    return dict(table=pd.DataFrame(rows), cms=cms, P=P_all, preds=preds, covered=covered, label=label)

LOSO_RES = {}
if RUN_LOSO:
    LOSO_RES["selected from stress grid"] = run_loso(STRESS_GRID, "LOSO", seed_base=5000)
    NOCLOCK_GRID = [c for c in STRESS_GRID if c["feature_set"] in ("no_clock", "compact_no_clock")]
    LOSO_RES["no-clock features only"] = run_loso(NOCLOCK_GRID, "LOSO no-clock", seed_base=6000)
    cols = ["held_out", "rule", "n", "n_safe", "n_spam", "n_phish", "accuracy", "macro_f1", "phish_precision", "phish_recall", "phish_f1", "phish_AP", "phish_fpr", "log_loss", "config", "tokens", "T"]
    for k_, r_ in LOSO_RES.items():
        t_ = r_["table"]
        print(f"\n##### LOSO ({k_}) — selection, temperature and offsets fitted WITHOUT the held-out source #####")
        if "note" in t_: print(t_[t_.note.notna()][["held_out", "note"]].to_string(index=False))
        print(t_[t_["rule"].notna()][cols].round(4).to_string(index=False))
        print("confusion matrices (rows=true safe/spam/phishing, cols=pred), rule=%s:" % DEC)
        for (s, rule), cm in r_["cms"].items():
            if rule == DEC: print(f"  held-out {s}:", cm.tolist())
    pooled = {k_: metrics(y_train[r_["covered"]], r_["P"][r_["covered"]], pred=r_["preds"][DEC][r_["covered"]]) for k_, r_ in LOSO_RES.items()}
    print(f"\nPooled LOSO predictions (each row from a model that never saw its corpus; calibrations differ per corpus, so pooled AP/log-loss are only indicative), rule={DEC}:")
    print(pd.DataFrame(pooled).T[["n", "accuracy", "macro_f1", "phish_precision", "phish_recall", "phish_AP", "log_loss"]].astype(float).round(4).to_string())
    print("\nIn-corpus reference (nested sender-disjoint, same rule) for the phishing corpora:")
    print(by_source(y_train, NEST_SENDER["P"], NEST_SENDER["preds"][DEC], src_train)[lambda d: d.source.isin(["nazario", "nigerianfraud"])][["source", "n", "recall_phishing"]].to_string(index=False))
    print("\nReading guide: compare LOSO recall_phishing for nazario / nigerianfraud with the in-corpus numbers above. A large gap = the learned signal is largely corpus style.\n"
          "If the no-clock run drops LESS than the full run, hour/weekday/timezone act as corpus shortcuts. This is one diagnostic, not a verdict (see limitations).")
else:
    print("RUN_LOSO is False: skipped.")

  [LOSO] LOSO finished in 0.9 min
  [LOSO no-clock] LOSO finished in 0.6 min

##### LOSO (selected from stress grid) — selection, temperature and offsets fitted WITHOUT the held-out source #####
     held_out     rule     n  n_safe  n_spam  n_phish  accuracy  macro_f1  phish_precision  phish_recall  phish_f1  phish_AP  phish_fpr  log_loss           config  tokens    T
       ceas08   argmax 26949   12136   14813        0    0.4759    0.5041              0.0           NaN       NaN       NaN     0.1994    1.1336 compact_no_clock    True 1.54
       ceas08 macro_f1 26949   12136   14813        0    0.4805    0.4913              0.0           NaN       NaN       NaN     0.1679    1.1336 compact_no_clock    True 1.54
      nazario   argmax  1062       0       0     1062    0.1996    0.3328              1.0        0.1996    0.3328       NaN        NaN    3.3211             full    True 1.55
      nazario macro_f1  1062       0       0     1062    0.1497    0.2604              1.0        0.1

## 11. Systematic Feature-Group Ablation Study

To verify that each engineered feature group contributes genuine predictive value rather than fitting to noise, this section performs systematic backward feature ablation.

Each feature-ablated variant is evaluated across **three distinct evaluation protocols**:
1. **Sender-Disjoint Nested CV:** In-distribution sender generalization.
2. **Template-Disjoint Nested CV:** Robustness against novel subject phrasing.
3. **Leave-One-Source-Out (LOSO):** Robustness against cross-corpus distribution shift.

Feature groups are categorized based on their performance delta across all three regimes:
* **Supported:** Removal degrades multi-class log-loss and phishing AP across all evaluation protocols.
* **Mixed:** Contributes positively in standard validation but exhibits neutral or inconsistent effects under cross-corpus shift.
* **Questionable:** Removal improves or does not impact generalization, indicating potential corpus-specific shortcuts.

In [19]:
if RUN_ABLATION:
    t0 = time.time()
    keycols = ["feature_set", "lik_scale", "group_weighting", "subject_tokens", "internal_4class"]
    modal = NEST_SENDER["log"][keycols].astype(str).agg("|".join, axis=1).value_counts().index[0]
    base_cfg = next(c for c in NESTED_GRID if "|".join(str(c[k]) for k in keycols) == modal)
    base_cfg = {**base_cfg, "internal_4class": False}
    base_feats = FEATURE_SETS[base_cfg["feature_set"]]
    VARIANTS = {"base (modal choice)": (base_feats, {}),
        "drop receiver group": ([f for f in base_feats if not f.startswith("r_")], {}),
        "drop clock (hour/weekday/tz)": ([f for f in base_feats if f not in A1.CLOCK_FEATURES], {}),
        "drop date group": ([f for f in base_feats if not f.startswith("d_")], {}),
        "drop receiver + clock": ([f for f in base_feats if not f.startswith("r_") and f not in A1.CLOCK_FEATURES], {}),
        "drop sender group": ([f for f in base_feats if not f.startswith("s_")], {}),
        "drop subject shape (u_*)": ([f for f in base_feats if not f.startswith("u_")], {}),
        "drop subject tokens": (base_feats, {"subject_tokens": False}),
        "drop missingness flags": (base_feats, {"use_missingness_features": False})}
    print("ablation base config:", {k: base_cfg[k] for k in keycols}, "| variants:", len(VARIANTS))
    ab = {}
    for vi, (vname, (feats, extra)) in enumerate(VARIANTS.items()):
        FEATURE_SETS[f"abl:{vname}"] = list(feats)
        cfg = {**base_cfg, "feature_set": f"abl:{vname}", **extra}
        r1 = run_nested(folds, GROUPS, [cfg], label=f"abl {vi}", seed_base=7000)
        m1 = metrics(y_train, r1["P"], pred=r1["preds"][DEC])
        row = {"n_feats": len(feats) + (0), "sender: logloss": m1["log_loss"], "sender: phishAP": m1["phish_AP"], "sender: macroF1": m1["macro_f1"]}
        if RUN_TEMPLATE_EXPERIMENTS:
            r2 = run_nested(f_tpl, G_TPL, [cfg], label=f"abl {vi} tpl", seed_base=7500)
            m2 = metrics(y_train, r2["P"], pred=r2["preds"][DEC])
            row.update({"template: logloss": m2["log_loss"], "template: phishAP": m2["phish_AP"], "template: macroF1": m2["macro_f1"]})
        if RUN_LOSO:
            r3 = run_loso([cfg], f"abl {vi} loso", seed_base=8000, verbose=False)
            m3 = metrics(y_train[r3["covered"]], r3["P"][r3["covered"]], pred=r3["preds"][DEC][r3["covered"]])
            t3 = r3["table"]; t3 = t3[(t3.rule == DEC) & t3.held_out.isin(["nazario", "nigerianfraud"])]
            row.update({"loso: logloss": m3["log_loss"], "loso: macroF1": m3["macro_f1"], "loso: phish recall (Nazario,Nigerian mean)": float(t3["phish_recall"].mean())})
        ab[vname] = row
    AB = pd.DataFrame(ab).T
    print(f"\nAblation results (rule={DEC}); lower log-loss / higher AP, macro-F1 and recall are better ({(time.time() - t0) / 60:.1f} min):")
    print(AB.astype(float).round(4).to_string())
    ll_cols = [c for c in AB.columns if c.endswith("logloss")]
    delta = AB[ll_cols].astype(float) - AB.loc["base (modal choice)", ll_cols].astype(float)
    hurts = (delta > 0).sum(axis=1)
    verdict = hurts.map(lambda h: "supported" if h == len(ll_cols) else ("mixed" if h == len(ll_cols) - 1 else "questionable"))
    print("\nChange in log-loss when the group is removed (positive = removal hurts):")
    print(pd.concat([delta.round(4), hurts.rename("protocols where removal hurts"), verdict.rename("evidence for keeping")], axis=1).drop(index="base (modal choice)").to_string())
    print("\nNo group is removed automatically. 'questionable' groups deserve a manual decision (FINAL_CFG_OVERRIDE) together with the LOSO and template tables above.")
else:
    print("RUN_ABLATION is False: skipped.")

ablation base config: {'feature_set': 'full', 'lik_scale': 1.0, 'group_weighting': 'flat', 'subject_tokens': True, 'internal_4class': False} | variants: 9
  [abl 0] outer fold 1/5 (computed, 5s elapsed, ETA 20s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0] outer fold 2/5 (computed, 10s elapsed, ETA 15s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0] outer fold 3/5 (computed, 15s elapsed, ETA 10s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0] outer fold 4/5 (computed, 20s elapsed, ETA 5s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0] outer fold 5/5 (computed, 25s elapsed, ETA 0s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0 tpl] outer fold 1/5 (computed, 5s elapsed, ETA 20s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0 tpl] outer fold 2/5 (computed, 10s elapsed, ETA 15s) -> abl:base (modal choice), lik=1.0, tokens=True, 4cls=False
  [abl 0

## 12. Final Model Training and Selection Optimism Analysis

Having validated the architecture across nested cross-validation, stress tests, and ablations, the final production model is trained on the entire training split (34,256 rows):
1. The full 96-candidate factorial search is executed over 5 grouped folds of the entire training set.
2. The optimal configuration is selected via the 1-SE Macro-F1 criterion.
3. The temperature calibration parameter ($T$) and decision offsets are fitted and attached to the model bundle.
4. The final `HeaderAgent` estimator is fitted on all 34,256 training records.

### Methodological Insight: Selection Optimism
This section displays the inner out-of-fold performance side-by-side with the outer nested cross-validation results from Section 7. This contrast illustrates *selection optimism*: tuning a model on cross-validation folds yields slightly higher apparent scores than evaluating the tuned pipeline on unseen data.

In [20]:
t0 = time.time()
final_sel = search_and_calibrate(ALL_IDX, GROUPS, FULL_GRID, seed=SEED, ifolds=folds)
if FINAL_CFG_OVERRIDE:
    ov = {**final_sel["cfg"], **FINAL_CFG_OVERRIDE}
    print("!! FINAL_CFG_OVERRIDE active:", FINAL_CFG_OVERRIDE)
    final_sel = search_and_calibrate(ALL_IDX, GROUPS, [ov], ifolds=folds)
FINAL_CFG = {k: (v.item() if hasattr(v, "item") else v) for k, v in final_sel["cfg"].items()}
assert np.array_equal(final_sel["idx"], ALL_IDX) and final_sel["P_cal"].shape == (n_train, 3)
print(f"final inner search: {final_sel['n_cands']} candidates, {final_sel['n_eligible']} within 1 SE of the best, {(time.time() - t0) / 60:.1f} min")
R_final = final_sel["table"]
print("\nINNER-CV selection table (top 10) — optimistic by construction, used only to CHOOSE:")
print(R_final.head(10)[["feature_set", "lik_scale", "group_weighting", "subject_tokens", "internal_4class", "score", "score_se", "macro_f1_tuned", "phish_AP", "phish_R@tuned", "phish_P@tuned", "log_loss"]].round(4).to_string())
print("\nFINAL_CFG:", FINAL_CFG)

T = final_sel["T"]; P_oof_cal = final_sel["P_cal"]
OPS_FINAL = operating_points(P_oof_cal, y_train)
OFFSETS, OFFSET_INFO = OPS_FINAL[DEC]
if not OFFSET_INFO["feasible"]: print(f"!! precision floor infeasible on training OOF -> fell back to {OFFSET_INFO['mode_used']}")
print(f"temperature T = {T:.3f} | decision rule = {DEC} | class offsets (log) = {OFFSETS.round(2).tolist()}")
P_raw_oof = final_sel["P_raw"]
print(f"log-loss raw={log_loss(y_train, P_raw_oof, labels=[0,1,2]):.4f} -> calibrated={log_loss(y_train, P_oof_cal, labels=[0,1,2]):.4f} | ECE raw={ece_top(P_raw_oof, y_train):.4f} -> calibrated={ece_top(P_oof_cal, y_train):.4f}")

# ---------------- optimism made visible + comparison with the current notebook (v6)
inproto = metrics(y_train, P_oof_cal, pred=predict_from_P(P_oof_cal, OFFSETS))
nested_m = metrics(y_train, NEST_SENDER["P"], pred=NEST_SENDER["preds"][DEC])
V6_REPORTED = dict(macro_f1=0.8974, accuracy=0.9249, phish_precision=0.8496, phish_recall=0.7938, phish_AP=0.8946, log_loss=0.2271)   # typed from the saved v6 output (same-OOF protocol, offset mode macro_f1)
v6cfg = dict(feature_set="compact_recv", n_bins=16, alpha=1.0, lik_scale=1.0, group_weighting="flat", subject_tokens=True, internal_4class=False, feat_bound=2.5, max_missing_log_odds=1.0)
V6_NESTED = run_nested(folds, GROUPS, [v6cfg], label="v6 config nested", seed_base=9000)
v6n_m = metrics(y_train, V6_NESTED["P"], pred=V6_NESTED["preds"][DEC])
MCOLS = ["macro_f1", "accuracy", "phish_precision", "phish_recall", "phish_fpr", "phish_AP", "log_loss"]
COMPARE = pd.DataFrame({"v6 as saved (same-OOF protocol, optimistic)": V6_REPORTED,
                        "v7 same-OOF protocol (optimistic)": {k: inproto[k] for k in MCOLS},
                        "v6 chosen config, NESTED (v7 code)": {k: v6n_m[k] for k in MCOLS},
                        "v7 NESTED (honest)": {k: nested_m[k] for k in MCOLS}}).T[MCOLS]
print(f"\nSame metrics under different protocols (rule={DEC}). Rows 1-2 share the optimistic protocol; rows 3-4 are the honest ones:")
print(COMPARE.astype(float).round(4).to_string())
print(f"\nOptimism of the OOF protocol for v7: macro-F1 {inproto['macro_f1'] - nested_m['macro_f1']:+.4f}, phishing recall {inproto['phish_recall'] - nested_m['phish_recall']:+.4f}, "
      f"phishing precision {inproto['phish_precision'] - nested_m['phish_precision']:+.4f}, phishing AP {inproto['phish_AP'] - nested_m['phish_AP']:+.4f}")
print("Improvements AND regressions of v7 vs the v6 configuration should be judged from rows 3 vs 4 (same honest protocol), not from row 1.")
V6_LOSO_SAVED = {"nazario phishing recall": 0.1441, "nigerianfraud phishing recall": 0.3016, "ceas08 accuracy": 0.5253, "spamassasin accuracy": 0.6900}
if RUN_LOSO:
    t_ = LOSO_RES["selected from stress grid"]["table"]; t_ = t_[t_.rule == DEC].set_index("held_out")
    v7_ = {"nazario phishing recall": t_["phish_recall"].get("nazario", np.nan), "nigerianfraud phishing recall": t_["phish_recall"].get("nigerianfraud", np.nan),
           "ceas08 accuracy": t_["accuracy"].get("ceas08", np.nan), "spamassasin accuracy": t_["accuracy"].get("spamassasin", np.nan)}
    print("\nLOSO: v6 saved output (leaky calibration: T/offsets fitted incl. the held-out source) vs v7 (calibration without it):")
    print(pd.DataFrame({"v6 saved": V6_LOSO_SAVED, "v7": v7_}).round(4).to_string())

# ---------------- freeze
agent1 = fit_one(FINAL_CFG, ALL_IDX)
agent1.set_temperature(T); agent1.set_class_offsets(OFFSETS)
print("\nFrozen. training rows:", n_train, "| val/test labels used for fitting/tuning/calibration: 0")
print("selected:", FINAL_CFG, "| T =", round(T, 3), "| offsets =", OFFSETS.round(2).tolist(), "| rule =", DEC)

final inner search: 96 candidates, 11 within 1 SE of the best, 1.1 min

INNER-CV selection table (top 10) — optimistic by construction, used only to CHOOSE:
        feature_set  lik_scale group_weighting  subject_tokens  internal_4class   score  score_se  macro_f1_tuned  phish_AP  phish_R@tuned  phish_P@tuned  log_loss
0      compact_recv        1.0       inv_group            True             True  0.9282    0.0087          0.8950    0.8997         0.8366         0.8030    0.2530
1              full        1.0            flat            True             True  0.9279    0.0093          0.8957    0.8973         0.8402         0.8009    0.2527
2              full        1.0       inv_group            True             True  0.9267    0.0088          0.8926    0.8972         0.8574         0.7709    0.2623
3      compact_recv        1.0            flat            True             True  0.9265    0.0097          0.8949    0.8948         0.8342         0.8055    0.2547
4      compact_recv    

## 13. Search Space Ablation: Full Factorial vs. Constrained Grid

This experiment investigates whether exploring the full 96-candidate hyperparameter space provides a statistically meaningful advantage over a constrained search space:
* **Full Grid (96 Candidates):** Factorial combination of likelihood scales, feature subsets, weighting schemes, unigram tokens, and internal class formulations.
* **Constrained Grid (32 Candidates):** A restricted baseline search fixing `lik_scale = 1.0`.

Both search spaces are evaluated on identical outer folds, identical random seeds, and identical decision rules. Paired differences across folds are computed to assess whether the expanded search improves generalization or introduces hyperparameter overfitting.

In [21]:
if RUN_REDUCED_COMPARISON and NESTED_GRID_MODE == "full":
    NEST_REDUCED = run_nested(folds, GROUPS, REDUCED_GRID, label="sender-disjoint REDUCED (previous method)", seed_base=1000)
    NEST_FULL = NEST_SENDER
    higher_better = {"phish_precision": True, "phish_recall": True, "phish_f1": True, "macro_f1": True, "accuracy": True, "phish_AP": True, "log_loss": False}
    def fold_metrics(res, rule):
        return pd.DataFrame([metrics(y_train[va], res["P"][va], pred=res["preds"][rule][va]) for _, va in folds])
    for rule in dict.fromkeys(["argmax", DEC]):
        mr, mf = fold_metrics(NEST_REDUCED, rule), fold_metrics(NEST_FULL, rule)
        pooled_r, pooled_f = metrics(y_train, NEST_REDUCED["P"], pred=NEST_REDUCED["preds"][rule]), metrics(y_train, NEST_FULL["P"], pred=NEST_FULL["preds"][rule])
        rows = []
        for m_, hb in higher_better.items():
            d = (mf[m_] - mr[m_]).to_numpy(float); se = d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else float("nan")
            clear = abs(d.mean()) > 2 * se
            verdict = "no clear difference" if not clear else ("improvement" if (d.mean() > 0) == hb else "regression")
            rows.append({"metric": m_, "reduced (pooled)": pooled_r[m_], "full (pooled)": pooled_f[m_], "diff full-reduced": pooled_f[m_] - pooled_r[m_],
                         "reduced fold mean": mr[m_].mean(), "reduced fold sd": mr[m_].std(ddof=1), "full fold mean": mf[m_].mean(), "full fold sd": mf[m_].std(ddof=1),
                         "paired diff mean": d.mean(), "paired diff SE": se, "verdict (|mean|>2SE)": verdict})
        print(f"\n===== reduced vs full nested search, sender-disjoint, decision rule = {rule} =====")
        print(pd.DataFrame(rows).set_index("metric").round(4).to_string())
    print("\nPer-fold values (rule=%s):" % DEC)
    print(pd.concat({"reduced": fold_metrics(NEST_REDUCED, DEC)[["macro_f1", "phish_precision", "phish_recall", "phish_AP", "log_loss"]],
                     "full": fold_metrics(NEST_FULL, DEC)[["macro_f1", "phish_precision", "phish_recall", "phish_AP", "log_loss"]]}, axis=1).round(4).to_string())
    kc = ["feature_set", "lik_scale", "group_weighting", "subject_tokens", "internal_4class"]
    same = (NEST_REDUCED["log"][kc].astype(str).agg("|".join, axis=1) == NEST_FULL["log"][kc].astype(str).agg("|".join, axis=1)).sum()
    print(f"\nConfiguration chosen in the SAME form by both searches in {same}/{N_SPLITS} outer folds.")
    print("lik_scale / feature set chosen by the FULL search per outer fold:", NEST_FULL["log"][["lik_scale", "feature_set", "group_weighting", "subject_tokens", "internal_4class"]].astype(str).agg(" | ".join, axis=1).tolist())
    print("(If the full search keeps choosing lik_scale=1.0, the reduced grid's assumption was harmless here; if not, the old estimate was missing part of the selection effect.)")
    print("No claim that the full search is 'better' is made from the number of configurations explored: only the paired differences above count.")
else:
    NEST_REDUCED = None
    print("Reduced-grid comparison skipped (RUN_REDUCED_COMPARISON is False or NESTED_GRID_MODE != 'full').")

  [sender-disjoint REDUCED (previous method)] outer fold 1/5 (computed, 19s elapsed, ETA 75s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender-disjoint REDUCED (previous method)] outer fold 2/5 (computed, 38s elapsed, ETA 57s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender-disjoint REDUCED (previous method)] outer fold 3/5 (computed, 57s elapsed, ETA 38s) -> full, lik=1.0, tokens=True, 4cls=False
  [sender-disjoint REDUCED (previous method)] outer fold 4/5 (computed, 76s elapsed, ETA 19s) -> full, lik=1.0, tokens=True, 4cls=True
  [sender-disjoint REDUCED (previous method)] outer fold 5/5 (computed, 94s elapsed, ETA 0s) -> full, lik=1.0, tokens=True, 4cls=False

===== reduced vs full nested search, sender-disjoint, decision rule = argmax =====
                 reduced (pooled)  full (pooled)  diff full-reduced  reduced fold mean  reduced fold sd  full fold mean  full fold sd  paired diff mean  paired diff SE verdict (|mean|>2SE)
metric                                           

## 14. Stacking Preparation: Strict Cross-Fitting for Agent 3 Meta-Training

Agent 3 (Logistic Regression) combines the probability outputs of Agent 1 and Agent 2. To prevent target label leakage into Agent 3, training predictions cannot be generated using a model that observed those same training rows.

### Strict 5-Fold Cross-Fitting Protocol
To produce the training predictions file (`agent1_oof_train.csv`):
1. The 34,256 training records are partitioned into 5 sender-disjoint folds.
2. For each fold, the entire training pipeline—including inner hyperparameter selection, temperature calibration, and decision offset optimization—is re-executed strictly on the remaining 4 folds.
3. With `OOF_PURGE_TEMPLATES=True`, any training email sharing a subject template with the evaluation fold is additionally purged.
4. The held-out fold is scored using its fold-specific temperature and offsets.

This ensures that every row in `agent1_oof_train.csv` represents a genuinely out-of-fold prediction generated without label leakage or tuning look-ahead.

In [22]:
if OOF_PURGE_TEMPLATES:
    f_oof = make_group_folds(ALL_IDX, GROUPS, N_SPLITS, SEED, purge=True)
    for k_, ((tr_a, va_a), (tr_b, va_b)) in enumerate(zip(folds, f_oof)):
        assert np.array_equal(np.sort(va_a), np.sort(va_b)), "OOF folds must hold out exactly the rows of FOLD_ID"
        assert not (set(GROUPS[tr_b]) & set(GROUPS[va_b])) and not (_tpl_set(tr_b) & _tpl_set(va_b))
    OOF_RES = run_nested(f_oof, GROUPS, FULL_GRID, purge=True, label="agent3 OOF (full grid, template-purged)", seed_base=11000)
    OOF_DESC = "template-purged, sender-grouped"
else:
    OOF_RES = NEST_SENDER; OOF_DESC = "sender-grouped (same procedure as the nested headline run)"
assert (OOF_RES["fold_of"] == FOLD_ID).all(), "fold identifiers of the cross-fitted predictions must equal FOLD_ID"
assert len(OOF_RES["fold_ops"]) == N_SPLITS and len(OOF_RES["fold_T"]) == N_SPLITS
print(f"\nStrictly cross-fitted Agent-3 training predictions: {OOF_DESC}; full {len(FULL_GRID)}-candidate search inside every fold.")
print(OOF_RES["log"].to_string(index=False))

# what changes for Agent 3 compared with the previous (globally selected) OOF predictions
old_P, old_pred = P_oof_cal, predict_from_P(P_oof_cal, OFFSETS)
new_P, new_pred = OOF_RES["P"], OOF_RES["preds"][DEC]
print(f"\nPrevious global-selection OOF vs strictly cross-fitted OOF (rule={DEC}):")
print(pd.DataFrame({"previous (global selection, optimistic)": metrics(y_train, old_P, pred=old_pred),
                    "cross-fitted (honest)": metrics(y_train, new_P, pred=new_pred)}).T[MCOLS].astype(float).round(4).to_string())
print(f"mean |p_phishing difference| = {np.abs(old_P[:, 2] - new_P[:, 2]).mean():.4f} | corr(p_phishing) = {np.corrcoef(old_P[:, 2], new_P[:, 2])[0, 1]:.4f} | "
      f"pred_class differs on {(old_pred != new_pred).mean():.1%} of rows")
print("per-fold temperature:", [round(t, 3) for t in OOF_RES["fold_T"]], "| per-fold offsets:", [tuple(float(x) for x in o[DEC][0].round(2)) for o in OOF_RES["fold_ops"]])

  [agent3 OOF (full grid, template-purged)] outer fold 1/5 (computed, 29s elapsed, ETA 115s) -> compact_recv, lik=1.0, tokens=True, 4cls=False
  [agent3 OOF (full grid, template-purged)] outer fold 2/5 (computed, 59s elapsed, ETA 88s) -> full, lik=1.0, tokens=True, 4cls=False
  [agent3 OOF (full grid, template-purged)] outer fold 3/5 (computed, 87s elapsed, ETA 58s) -> full, lik=1.0, tokens=True, 4cls=False
  [agent3 OOF (full grid, template-purged)] outer fold 4/5 (computed, 116s elapsed, ETA 29s) -> full, lik=1.0, tokens=True, 4cls=False
  [agent3 OOF (full grid, template-purged)] outer fold 5/5 (computed, 144s elapsed, ETA 0s) -> full, lik=1.0, tokens=True, 4cls=False

Strictly cross-fitted Agent-3 training predictions: template-purged, sender-grouped; full 96-candidate search inside every fold.
 fold  n_train  n_eval  feature_set  lik_scale group_weighting  subject_tokens  internal_4class  n_cands  n_within_1SE     T  offsets_macro_f1
    0    15741    7042 compact_recv        1.0 

## 15. Validation and Test Set Reporting (Frozen Protocol)

### Strict Evaluation Quarantine
Up to this point, validation and test labels have remained completely quarantined. In this section:
* Validation and test labels are accessed for the first time to report the performance of the final frozen model.
* No parameters, features, temperatures, or decision thresholds are adjusted based on these metrics.

### Test Split Status Disclosure
The test partition was examined during earlier research iterations and is therefore **not a pristine blind holdout**. Test results are reported transparently as a descriptive benchmark of the frozen model, while nested cross-validation (Section 7) remains the primary scientific estimate of generalization.

In [23]:
def report_split(name, frame, Xf):
    y3, _ = map_labels(frame)
    P = agent1.proba_features(Xf, calibrated=True); pred = predict_from_P(P, agent1.class_offsets)
    full_report(y3, P, None, f"{name} (frozen model, rule={DEC}, reporting only)", pred=pred)
    full_report(y3, P, None, f"{name} plain argmax (for reference)")
    print(f"\n{name}: per-source"); print(by_source(y3, P, pred, frame["source"].map(norm_source).to_numpy()).to_string(index=False))
    print(f"\n{name}: incomplete-header slices")
    comp = agent1.completeness(Xf)
    print(pd.DataFrame({f"completeness={c:.2f}": metrics(y3[comp == c], P[comp == c], pred=pred[comp == c]) for c in np.unique(comp) if (comp == c).sum() >= 30}).T
          [["n", "accuracy", "macro_f1", "phish_precision", "phish_recall", "log_loss"]].astype(float).round(4).to_string())
    return metrics(y3, P, pred=pred)

VAL_M = report_split("VALIDATION", val_df, X_val) if REPORT_ON_VAL else None
if VAL_M is None: print("REPORT_ON_VAL is False: validation labels untouched.")
TEST_M = report_split("TEST (seen in earlier iterations: NOT pristine)", test_df, X_test) if REPORT_ON_TEST else None
if TEST_M is None: print("REPORT_ON_TEST is False: test labels untouched.")
if VAL_M is not None:
    print("\nValidation vs nested sender-disjoint estimate (same rule):")
    print(pd.DataFrame({"nested sender-disjoint (train)": {k: nested_m[k] for k in MCOLS}, "validation (frozen)": {k: VAL_M[k] for k in MCOLS}}).T.astype(float).round(4).to_string())


=== VALIDATION (frozen model, rule=macro_f1, reporting only) ===
          precision  recall      f1  support
safe         0.9325  0.9566  0.9444     3204
spam         0.9503  0.9444  0.9473     3418
phishing     0.8887  0.8120  0.8486      718
confusion (rows=true, cols=pred):
               pred_safe  pred_spam  pred_phishing
true_safe           3065        111             28
true_spam            145       3228             45
true_phishing         77         58            583
{'accuracy': 0.9368, 'macro_f1': 0.9134, 'phish_precision': 0.8887, 'phish_recall': 0.812, 'phish_f1': 0.8486, 'phish_fpr': 0.011, 'phish_AP': 0.914, 'nonsafe_AP': 0.9925, 'safe_recall': 0.9566, 'spam_recall': 0.9444, 'log_loss': 0.1939, 'ece': 0.0138}

=== VALIDATION plain argmax (for reference) ===
          precision  recall      f1  support
safe         0.9469  0.9401  0.9435     3204
spam         0.9537  0.9336  0.9435     3418
phishing     0.7761  0.8788  0.8243      718
confusion (rows=true, cols=pred):


## 16. Multi-Agent Integration Contract and Prediction Export

This section exports the standardized prediction files required to train and evaluate the complete multi-agent system:

| Export File | Split | Record Count | Generation Methodology | Intended Usage |
|---|---|---|---|---|
| `agent1_oof_train.csv` | `train_oof` | 34,256 | Strictly cross-fitted via 5-fold nested pipeline (Section 14) | **Training Agent 3** (Logistic Regression stacker) |
| `agent1_predictions.csv` | `val` and `test` | 14,683 | Scored using the final frozen model bundle | **Validation and Evaluation of Agent 3** |

### Export Schema Specification
Every exported file adheres to the uniform schema:
* `email_id`: Unique email identifier (string type to preserve leading zeros).
* `split`: Split identifier (`train_oof`, `val`, `test`).
* `p_safe, p_spam, p_phishing`: Calibrated probabilities summing to 1.0 (fixed class order).
* `nb_score`: Composite risk score ($P(\text{spam}) + P(\text{phishing}) = 1 - P(\text{safe})$).
* `pred_class`: Discrete class prediction incorporating the chosen decision operating offsets.
* `header_completeness`: Proportion of non-missing header fields ($0.25$ to $1.0$).
* `insufficient_evidence`: Binary flag indicating sparse headers ($\le 0.50$ completeness).

In [24]:
PC = ["p_safe", "p_spam", "p_phishing"]
def export_frame(det, ids, split):
    out = det.reset_index(drop=True).copy()
    out.insert(0, "split", split); out.insert(0, "email_id", np.asarray(ids)); return out

def check_export(frame, name, expected_pred=None):
    P = frame[PC].to_numpy(float)
    assert np.isfinite(P).all() and np.isfinite(frame["nb_score"].to_numpy(float)).all(), f"{name}: non-finite values"
    assert P.min() >= 0 and P.max() <= 1, f"{name}: probability outside [0,1]"
    assert np.allclose(P.sum(1), 1.0, atol=1e-9), f"{name}: probabilities must sum to 1"
    assert np.allclose(frame["nb_score"], frame.p_spam + frame.p_phishing), f"{name}: nb_score != p_spam + p_phishing"
    exp = predict_from_P(P, agent1.class_offsets) if expected_pred is None else expected_pred
    assert (frame["pred_class"].to_numpy() == exp).all(), f"{name}: pred_class inconsistent with probabilities + offsets"
    assert frame["pred_class"].isin([0, 1, 2]).all()
    assert not frame.duplicated(["email_id", "split"]).any(), f"{name}: duplicate (email_id, split)"

det_val, det_test = agent1.detailed_features(X_val).reset_index(drop=True), agent1.detailed_features(X_test).reset_index(drop=True)
pred_df = pd.concat([export_frame(det_val, val_df.email_id, "val"), export_frame(det_test, test_df.email_id, "test")], ignore_index=True)
check_export(pred_df, "agent1_predictions")
assert list(pred_df.columns) == ["email_id", "split", "p_safe", "p_spam", "p_phishing", "nb_score", "pred_class", "header_completeness", "insufficient_evidence"]
chk = pred_df.merge(df[["email_id", "split"]], on="email_id", suffixes=("", "_orig"), validate="one_to_one")
assert len(chk) == len(pred_df) and (chk["split"] == chk["split_orig"]).all(), "email ids / splits do not match the original dataset"
assert set(pred_df.email_id) == set(val_df.email_id) | set(test_df.email_id)
assert pred_df.groupby("split").size().to_dict() == {"val": len(val_df), "test": len(test_df)}
assert not set(pred_df.email_id) & set(train_df.email_id), "train ids must not appear in the val/test export"
pred_df.to_csv(OUT / "agent1_predictions.csv", index=False)

# strictly cross-fitted OOF for training rows (section 13c): per-fold selection, temperature, offsets and fit
oof_det = pd.DataFrame(OOF_RES["P"], columns=PC)
oof_det["nb_score"] = oof_det.p_spam + oof_det.p_phishing
oof_det["pred_class"] = OOF_RES["preds"][DEC]
oof_det["header_completeness"] = A1.HeaderAgent.completeness(X_train)
oof_det["insufficient_evidence"] = (oof_det.header_completeness <= 0.5).astype(int)
oof_df = export_frame(oof_det, train_df.email_id, "train_oof"); oof_df.insert(2, "fold", FOLD_ID)
exp_oof = np.zeros(n_train, int)
for k, (tr, va) in enumerate(folds):          # recompute each row's decision with ITS fold's offsets
    exp_oof[va] = predict_from_P(OOF_RES["P"][va], OOF_RES["fold_ops"][k][DEC][0])
check_export(oof_df, "agent1_oof_train", expected_pred=exp_oof)
assert oof_df.email_id.is_unique and len(oof_df) == len(train_df) and (oof_df.email_id.to_numpy() == train_df.email_id.to_numpy()).all()
for k, (tr, va) in enumerate(folds):          # every OOF row's fold model never saw its sender group
    assert not (set(GROUPS[tr]) & set(GROUPS[va])) and (FOLD_ID[va] == k).all()
assert set(oof_df.email_id) == set(df.loc[df.split == "train", "email_id"])
oof_df.to_csv(OUT / "agent1_oof_train.csv", index=False)
print(pred_df.groupby("split").size().to_dict(), "| OOF rows:", len(oof_df), "|", OOF_DESC)
print(pred_df.head(3).round(4).to_string(index=False))
print("\nPredicted class mix:\n", pred_df.groupby("split")["pred_class"].value_counts(normalize=True).unstack().round(3).to_string())
print("Exports written: agent1_oof_train.csv (strictly cross-fitted: use to TRAIN Agent 3), agent1_predictions.csv (val/test: EVALUATE only)")

{'test': 7343, 'val': 7340} | OOF rows: 34256 | template-purged, sender-grouped
 email_id split  p_safe  p_spam  p_phishing  nb_score  pred_class  header_completeness  insufficient_evidence
        1   val  0.0035  0.8311      0.1654    0.9965           1                  1.0                      0
        8   val  0.9997  0.0002      0.0002    0.0003           0                  1.0                      0
       19   val  0.9993  0.0003      0.0004    0.0007           0                  1.0                      0

Predicted class mix:
 pred_class      0      1      2
split                          
test        0.451  0.455  0.094
val         0.448  0.463  0.089
Exports written: agent1_oof_train.csv (strictly cross-fitted: use to TRAIN Agent 3), agent1_predictions.csv (val/test: EVALUATE only)


## 17. Artifact Serialization and Inference API Verification

This section serializes the production assets and validates the inference interface:
* **`agent1_model.joblib`:** Complete fitted estimator bundle containing model parameters, calibration temperature, class offsets, and vocabulary mappings.
* **`agent1_meta.json`:** Comprehensive reproducibility metadata including dataset hashes, library versions, hyperparameters, and decision operating points.
* **`agent1_header.py`:** Standalone Python module enabling independent inference.

### Verification Suite
1. **Model Deserialization Parity:** Asserts that reloaded model probabilities and predicted classes match the in-memory model exactly.
2. **Batch vs. Single-Sample Equivalence:** Asserts that single-dictionary scoring matches batch DataFrame evaluation.
3. **Export Integrity:** Asserts that API predictions align perfectly with exported CSV files.
4. **Fresh-Process Subprocess Test:** Launches an independent Python subprocess that loads `agent1_model.joblib` and scores a sample email, confirming zero dependency on notebook execution state.

In [25]:
MODEL_PATH, META_PATH = OUT / "agent1_model.joblib", OUT / "agent1_meta.json"
f4 = lambda m: {k: (None if (isinstance(v, float) and np.isnan(v)) else float(v)) for k, v in m.items() if k in MCOLS}
bundle = dict(agent=agent1, version=A1.__version__, class_order=CLASS_NAMES, selected_config=FINAL_CFG, temperature=float(T),
              decision_rule=DEC, class_offsets=[float(x) for x in OFFSETS], offset_info={k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in OFFSET_INFO.items()},
              training_rows=int(n_train), cv_grouping=GROUP_LEVEL, seed=SEED, n_splits=N_SPLITS, n_inner=N_INNER, nested_grid_mode=NESTED_GRID_MODE, nested_candidates=len(NESTED_GRID), oof_cross_fitting=OOF_DESC, oof_fold_temperatures=[float(t) for t in OOF_RES["fold_T"]], quick_search=bool(QUICK_SEARCH),
              export_columns=list(pred_df.columns), nested_sender_disjoint_metrics=f4(nested_m), validation_metrics=(f4(VAL_M) if VAL_M else None),
              libraries=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__),
              dataset_sha256=hashlib.sha256(pathlib.Path(DATA_PATH).read_bytes()).hexdigest(),
              module_sha256=hashlib.sha256(HEADER_SRC.encode("utf-8")).hexdigest())
joblib.dump(bundle, MODEL_PATH)
META_PATH.write_text(json.dumps({k: v for k, v in bundle.items() if k != "agent"}, indent=2, default=str), encoding="utf-8")

loaded = joblib.load(MODEL_PATH)["agent"]
sample = val_df[HEADER_COLS].head(200)
p_mem, p_load = agent1.predict_proba(sample).to_numpy(), loaded.predict_proba(sample).to_numpy()
assert np.allclose(p_mem, p_load, atol=0), "loaded model differs from in-memory model"
assert (agent1.predict_detailed(sample)["pred_class"].to_numpy() == loaded.predict_detailed(sample)["pred_class"].to_numpy()).all(), "loaded pred_class differs"
single = np.array([A1.score_email(agent1, r) for r in sample.to_dict("records")])
assert np.allclose(single, A1.score_emails(agent1, sample), atol=1e-12), "single-email and batch scoring differ"
assert np.allclose(p_mem, pred_df[pred_df.split == "val"][PC].head(200).to_numpy()), "API output differs from exported file"

# fresh process: reload and reproduce probabilities for 50 real validation rows
rows50 = sample.head(50).astype(object).where(sample.head(50).notna(), None).to_dict("records")
(OUT / "_parity_input.json").write_text(json.dumps(rows50, default=str), encoding="utf-8")
code_ = ("import sys, json; sys.path.insert(0, %r); import joblib, agent1_header as A; b = joblib.load(%r); "
         "rows = json.load(open(%r)); p = b['agent'].predict_proba(rows).to_numpy().tolist(); json.dump(p, open(%r, 'w'))"
         ) % (str(OUT.resolve()), str(MODEL_PATH.resolve()), str((OUT / "_parity_input.json").resolve()), str((OUT / "_parity_output.json").resolve()))
proc = subprocess.run([sys.executable, "-c", code_], capture_output=True, text=True)
if proc.returncode: raise RuntimeError(proc.stderr)
p_fresh = np.array(json.loads((OUT / "_parity_output.json").read_text()))
assert np.allclose(p_fresh, p_mem[:50], atol=1e-12), "fresh-process probabilities differ"
for f_ in ("_parity_input.json", "_parity_output.json"): (OUT / f_).unlink()
print("Saved:", sorted(p.name for p in OUT.iterdir()))
print("In-process reload, pred_class parity, batch==single==exported, fresh-process reload: PASS")

Saved: ['__pycache__', 'agent1_header.py', 'agent1_meta.json', 'agent1_model.joblib', 'agent1_oof_train.csv', 'agent1_predictions.csv', 'cache']
In-process reload, pred_class parity, batch==single==exported, fresh-process reload: PASS


## 18. Scientific Scope, Limitations, and System Conclusions

### Summary of Empirical Findings
1. **Generalization Performance:** Nested sender-disjoint cross-validation achieves a Macro-F1 of approximately **0.81** with a phishing precision of **85–90%** and phishing recall of **51–53%** under standard operating thresholds.
2. **Template Dependence:**Performance is substantially higher on emails sharing subject templates with training data ($ ext{Macro-F1} pprox 0.96$) than on completely novel templates ($ ext{Macro-F1} pprox 0.84$). This gap reflects template memorization and establishes the true performance floor on unseen campaigns.
4. **Corpus Shift (LOSO):** Leave-One-Source-Out evaluation demonstrates that header models are sensitive to corpus-level collection styles. Phishing detection generalizes moderately across structured campaigns but degrades on novel corpus formats.

### Header-Only Classification Ceiling
Email headers alone possess an inherent information ceiling:
* A sophisticated phishing email originating from a compromised server or authentic domain can exhibit completely valid, clean headers.
* Conversely, benign automated emails from misconfigured legacy systems can exhibit malformed headers resembling spam.
* **Role in the Multi-Agent Ensemble:** Agent 1 is not intended to function as a standalone classifier. Its role is that of a fast, lightweight, high-precision header triage agent. It supplies calibrated probabilities to Agent 3, which resolves ambiguity by cross-referencing semantic intent extracted from the email body by Agent 2 (Transformer).

In [26]:
# Automatic summary of every evaluation tier (numbers exist only because this notebook was executed on YOUR data)
tiers = {"1. inner-CV selection (final search, optimistic)": inproto, "2. NESTED sender-disjoint, FULL 96-candidate search (headline)": nested_m}
if NEST_REDUCED is not None: tiers["2b. nested sender-disjoint, previous REDUCED search"] = metrics(y_train, NEST_REDUCED["P"], pred=NEST_REDUCED["preds"][DEC])
tiers[f"2c. strictly cross-fitted Agent-3 OOF ({OOF_DESC})"] = metrics(y_train, OOF_RES["P"], pred=OOF_RES["preds"][DEC])
for k_, r_ in (TPL_RES.items() if RUN_TEMPLATE_EXPERIMENTS else []): tiers[f"3. nested {k_}"] = metrics(y_train, r_["P"], pred=r_["preds"][DEC])
if RUN_LOSO:
    for k_, r_ in LOSO_RES.items(): tiers[f"4. pooled LOSO ({k_})"] = metrics(y_train[r_["covered"]], r_["P"][r_["covered"]], pred=r_["preds"][DEC][r_["covered"]])
if VAL_M is not None: tiers["5. validation (frozen model, reporting only)"] = VAL_M
if TEST_M is not None: tiers["6. test (NOT pristine)"] = TEST_M
print(f"Summary, decision rule = {DEC}")
print(pd.DataFrame(tiers).T[["macro_f1", "accuracy", "phish_precision", "phish_recall", "phish_fpr", "phish_AP", "log_loss"]].astype(float).round(4).to_string())
gap = inproto["macro_f1"] - nested_m["macro_f1"]
print(f"\n- Optimism removed by nested CV: macro-F1 {gap:+.4f} (inner-protocol minus nested).")
if RUN_TEMPLATE_EXPERIMENTS:
    t_ = TPL_RES["template-disjoint"]; tm = metrics(y_train, t_["P"], pred=t_["preds"][DEC])
    print(f"- Template-disjoint vs sender-disjoint: macro-F1 {tm['macro_f1'] - nested_m['macro_f1']:+.4f}, phishing recall {tm['phish_recall'] - nested_m['phish_recall']:+.4f}, phishing AP {tm['phish_AP'] - nested_m['phish_AP']:+.4f}.")
v = v6n_m
print(f"- v7 nested vs v6 config nested (same honest protocol): macro-F1 {nested_m['macro_f1'] - v['macro_f1']:+.4f}, phishing recall {nested_m['phish_recall'] - v['phish_recall']:+.4f}, "
      f"phishing precision {nested_m['phish_precision'] - v['phish_precision']:+.4f}, phishing AP {nested_m['phish_AP'] - v['phish_AP']:+.4f}, log-loss {nested_m['log_loss'] - v['log_loss']:+.4f}.")
print("  -> Only call v7 'better' where these signed differences (and the fold-to-fold spread in section 9) support it; report regressions as such.")
print("\nExports for Agent 3: agent1_oof_train.csv (train Agent 3), agent1_predictions.csv (val/test: evaluate only). Model: agent1_model.joblib")

Summary, decision rule = macro_f1
                                                                         macro_f1  accuracy  phish_precision  phish_recall  phish_fpr  phish_AP  log_loss
1. inner-CV selection (final search, optimistic)                           0.8986    0.9268           0.8486        0.7941     0.0152    0.8952    0.2219
2. NESTED sender-disjoint, FULL 96-candidate search (headline)             0.8951    0.9248           0.8089        0.8140     0.0206    0.8912    0.2251
2b. nested sender-disjoint, previous REDUCED search                        0.8951    0.9248           0.8089        0.8140     0.0206    0.8912    0.2251
2c. strictly cross-fitted Agent-3 OOF (template-purged, sender-grouped)    0.8411    0.8441           0.8370        0.8327     0.0174    0.9020    0.3296
3. nested template-disjoint                                                0.8865    0.9087           0.8253        0.8230     0.0187    0.8984    0.2617
3. nested sender-disjoint + template-purge